# 62 · Beyond 1:1 orthologs: paralog families and species-specific genes along the PT coordinate

The cross-species analyses (notebooks 03–61) work in a shared ortholog space built by
`cross_species.build_one_to_one_ortholog_map`. Genes in one-to-many, many-to-one and many-to-many
families, and genes without an ortholog, are outside that space. This notebook asks whether those
genes carry PT positional biology that the shared space cannot see. It analyses each species on its
own panel, along the paper's PT coordinate (SCF13), without any ortholog mapping for stage 1.

**Summary (numbers from this run).**
- **Stage 1 (inventory).** Within each species, along SCF13, with two specimens per species:
  - *Mouse:* non-1:1 genes are 6% of 10,730 PT-expressed probe-clean genes but 11% (68) of the 612
    strongly zonated ones (odds ratio 2.1, expression-matched 2.3, p < 0.001; 1.7–3.2 across support,
    pseudocount, probe and construction-gene variants). 42 of the 68 are outside the paper's shared space.
    They are mostly rodent-expanded drug-handling families (Cyp2d, Ugt2b, Ces1, Akr1c, Nat8f, Cyp2j,
    Slco1a, Slc22a OATs, Cyp4a) whose human members are flat or not PT-expressed.
  - *Human:* non-1:1 genes are 7% of PT-expressed and 6% (18) of the 304 strongly zonated genes (odds
    ratio 0.86, not enriched).
  - 20 orthogroups have a zonated, probe-clean member in both species and 22 have ≥ 2 zonated, probe-clean
    paralogs in one species, so stage 2(a–c) ran. Only 5 mouse and 9 human genes without any HCOP ortholog
    are zonated and probe-clean (two of the mouse five are annotation gaps), so the niche analysis 2(d)
    did not run.
- **Paralogs do not partition the axis; they co-position.** Paralog peaks are closer than matched random
  zonated pairs (mean separation 0.23 against 0.36, two-sided p = 0.02) and their shapes more alike
  (1 − r 0.61 against 0.93, p = 0.014). 17 of 58 pairs separate reproducibly in both specimens (matched
  random pairs 49%). Segment means alone separate 16 of those 17; the one within-segment separation is
  Cyp4a12a/Cyp4a14 inside mouse S2. Probe similarity does not explain the co-positioning, but the test
  loses significance when pairs with a probe within 8 mismatches of the paralog are dropped (post hoc).
- **Across species, families are not more divergent than 1:1 genes:** 6 of 20 family units differ in
  shape (0 under relabeling) against a matched 1:1 expectation of 7.3 (odds ratio 1.24, p = 0.47).
  Single-copy genes are not the union of their co-orthologs' positions. The positionally best-matching
  co-ortholog is the sequence-closest no more often than chance (14 of 26 against 12.7).
- **Physiological caveat that changes the reading.** Strongly zonated mouse non-1:1 genes are more often
  male-biased in mouse PT than strongly zonated 1:1 genes (27% against 16%, p = 0.025, post hoc; Xiong et
  al. 2023). Examples are Slco1a1, Cyp2j13, Cyp4a12a, Ugt2b37, Ugt2b38 and Kap. All specimens are male.
- **The shared space keeps reciprocal-best pairs inside families.** In 59 of 393 mouse families with a
  PT-expressed member, the kept mouse partner is not the family's most expressed PT member; in 29 it is
  not PT-expressed at all. One example is SLCO1A2–Slco1a5, while Slco1a1 and Slco1a6 carry the PT
  signal.

Cohort: 2 male control mice (whole-kidney sections) and 2 cortex sections from 1 male human donor.
Specimens, not structures, are replicates. Everything here is exploratory and descriptive. The
protocol was written before any expression result (workstream C plan, `plan.md`).

## 0 · Protocol, fixed before any result in this notebook

| Item | Rule |
|---|---|
| Structures | the 12,866 PT structures of `results/pt_reconstruction_v2/primary_pt_coordinate.csv` (SCF13), joined to the species-native matrices on `specimen_obsname`; common support = inside every specimen's 1st–99th percentile (notebook 37) |
| Expression | CP10k against each structure's total over its own species panel (linear scale) |
| Orthogroups | connected components of HCOP human–mouse pairs with ≥ 3 supporting databases (the paper map's rule); sensitivity ≥ 2 and ≥ 5. Classes 1:1, 1:many, many:1, many:many, no_consensus (only low-support pairs), no_ortholog (no HCOP pair) |
| Probe status | every included probe aligned (bowtie2, gapless, ≤ 5 mismatches) to the transcriptome its panel was designed on (mouse Ensembl 98; human Ensembl 110, see deviations); off-target = another gene on the probe strand, not an overlapping locus, not a pseudogene; probe `cross` ≤ 3 mismatches, `near` 4–5; gene `ambiguous` if any probe is `cross`; **probe-clean = clean or near** (strict sensitivity: clean only) |
| PT-expressed | detected (≥ 1 count) in ≥ 2% of structures in both specimens of the species |
| Curve | per specimen, library-weighted least-squares cubic B-spline of CP10k (notebook 12 basis, 6 df, shared knots), 61-point common grid; φ = log2(curve + 0.05) |
| Zonated | weighted F-test (spline vs intercept) BH q < 0.01 in both specimens, amplitude max φ − min φ ≥ 1 in both, between-specimen Pearson(φ) ≥ 0.5; strength Z = min amplitude; strong: Z ≥ 2 |
| Stop rule | stop after stage 1 if fewer than 10 non-1:1 orthogroups have a PT-expressed, zonated, probe-clean member in **each** species; stage 2(b) test needs ≥ 8 families with ≥ 2 such paralogs in one species; stage 2(d) needs ≥ 10 zonated probe-clean genes without an ortholog |
| Stage 2(a) | family unit = summed CP10k of PT-expressed probe-clean members; call when both within-species shape distances are the smallest of the six specimen pairs and the cross − within gap ≥ 0.5 (RMS log2 of centred profiles); balanced relabelings as the control; 1:1 comparator matched on expression and Z quintiles |
| Stage 2(b) | paralog pairs (zonated, probe-clean, same species and orthogroup): peak separation and 1 − r against 1,000 matched random pairs (quintiles of expression and Z, outside the family; seed 62); family score = mean percentile; two-sided Wilcoxon against 0.5 and a global matched-draw test; separated = per-specimen peaks ≥ 0.2 apart in the same order in both specimens; NNLS union fit of the single-copy gene |
| Stage 2(c) | best expression match among co-orthologs (mean cross-species Pearson of φ) against the Ensembl Compara 98 sequence-closest co-ortholog; chance = Σ 1/k |
| Stage 2(d) | zonated, probe-clean genes without an ortholog; functional class from the repo's pathway libraries (human symbols) and symbol root; niche = same root or pathway in the other species with a peak within 0.15 |
| Flags | fasting/PPARα-responsive and androgen-regulated mouse genes; construction genes of the coordinate (611 PCA genes, notebook 36) |

In [ ]:
import argparse
import json
import os
import shutil
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--threads', type=int)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
THREADS = int(args.threads or os.environ.get('PSEUDOSPACE_THREADS') or 16)
output = results_root / 'pt_non_one_to_one_orthologs'
tables, figures = output / 'tables', output / 'figures'
alignment_dir = output / 'probe_alignment'
for folder in (tables, figures, alignment_dir):
    folder.mkdir(parents=True, exist_ok=True)
cache_root = output / 'stage_cache'
external = data_root / 'external'
COORDINATE = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'
PAPER_MAP = results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'
NB36_CACHE = results_root / 'pt_reconstruction_v2' / 'nb36' / 'stage_cache'
HCOP = data_root / 'human_mouse_hcop_fifteen_column.txt.gz'
PROBES = {'mouse': external / 'visium_probe_sets' / 'Visium_Mouse_Transcriptome_Probe_Set_v2.0_mm10-2020-A.csv',
          'human': external / 'visium_probe_sets' / 'Visium_Human_Transcriptome_Probe_Set_v2.0_GRCh38-2020-A.csv'}
MOLECULE_INFO = [data_root / s / 'outs' / 'molecule_info.h5' for s in ('HUK_COR_1', 'HUK_MED_1')]
TRANSCRIPTS = {'mouse': [external / 'ensembl_r98_transcripts' / f for f in
                         ('Mus_musculus.GRCm38.cdna.all.fa.gz', 'Mus_musculus.GRCm38.ncrna.fa.gz')],
               'human': [external / 'ensembl_r110_human_transcripts' / f for f in
                         ('Homo_sapiens.GRCh38.cdna.all.fa.gz', 'Homo_sapiens.GRCh38.ncrna.fa.gz')]}
TRANSCRIPTOME_RELEASE = {'mouse': 'Ensembl 98 (mm10-2020-A)', 'human': 'Ensembl 110 (GRCh38-2024-A)'}
COMPARA = external / 'ensembl_compara_human_mouse' / 'Compara.98.protein_default.homologies.human_mouse.tsv.gz'
SPECIMENS = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
MATRICES = {s: data_root / 'tubule_by_gene' / f'{s}_tubule_by_gene_caleb.h5ad' for v in SPECIMENS.values() for s in v}
required = [COORDINATE, PAPER_MAP, HCOP, COMPARA, *PROBES.values(), *MOLECULE_INFO, *MATRICES.values(),
            *[p for v in TRANSCRIPTS.values() for p in v]]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')
if shutil.which('bowtie2') is None or shutil.which('bowtie2-build') is None:
    raise RuntimeError('bowtie2 and bowtie2-build must be on PATH for the probe audit.')

NOTEBOOK_LOGIC_VERSION = '62.non_one_to_one.1'  # Bump when a cached calculation changes.
PRIMARY_SUPPORT, SUPPORT_RULES = 3, (2, 3, 5)
DETECTION, Q_MAX, MIN_AMPLITUDE, MIN_REPLICATION, STRONG = .02, .01, 1., .5, 2.
PSEUDOCOUNT, PSEUDOCOUNT_SENSITIVITY = .05, .2
GRID_POINTS, BASIS_DF = 61, 6
CROSS_MM, NEAR_MM = 3, 5
STOP_FAMILIES, MIN_PARTITION_FAMILIES, MIN_SPECIFIC = 10, 8, 10
SEPARATION_GAP, PEAK_SEPARATION, NICHE_WINDOW = .5, .2, .15
N_DRAWS, SEED = 1000, 62
# Flags from notebook 44's PPARα program and the androgen-regulated genes named in the protocol.
FASTING_PPARA = {'Cyp4a10', 'Cyp4a14', 'Acot1', 'Hmgcs2', 'Pdk4', 'Ehhadh', 'Cpt1a', 'Acox1', 'Crot', 'Nudt19'}
ANDROGEN = {'Cyp4a12a', 'Cyp4a12b', 'Slco1a1', 'Kap'}
print('Results folder:', output)

## 1 · PT structures and species-native expression

The coordinate file is joined to each specimen's native matrix. The join must be exact: every
coordinate structure exists in its matrix, and identifiers are unique on both sides.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

from pseudospace import paralog_zonation as pz
from pseudospace.coordinate_inputs import common_support_mask
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import bh_adjust, gam_internal_knots, make_gam_design

coordinate = pd.read_csv(COORDINATE)
if not coordinate.structure_id.is_unique:
    raise ValueError('Duplicated structure ids in the coordinate file.')
expected = {sp: sorted(v) for sp, v in SPECIMENS.items()}
observed = coordinate.groupby('species').specimen.apply(lambda v: sorted(v.unique())).to_dict()
if observed != expected:
    raise ValueError(f'Unexpected cohort in the coordinate file: {observed}')
if set(coordinate.segment_class) != {'PT-S1', 'PT-S2', 'PT-S3'} or not np.isfinite(coordinate.position).all():
    raise ValueError('Expected PT-S1/S2/S3 labels and finite positions.')

native, join_rows = {}, []
for species, names in SPECIMENS.items():
    for name in names:
        rows = coordinate[coordinate.specimen.eq(name)]
        a = ad.read_h5ad(MATRICES[name])
        if not (a.obs_names.is_unique and a.var_names.is_unique):
            raise ValueError(f'{name}: duplicated identifiers in the native matrix.')
        ids = rows.structure_id.str.slice(len(name) + 1)
        if not rows.structure_id.str.startswith(name + '_').all():
            raise ValueError(f'{name}: coordinate ids lack the specimen prefix.')
        missing = ~ids.isin(a.obs_names)
        if missing.any():
            raise ValueError(f'{name}: {int(missing.sum())} coordinate structures missing from the matrix.')
        a = a[ids.to_numpy()].copy()
        X = sparse.csr_matrix(a.X)
        if (X.data < 0).any() or not np.allclose(X.data, np.round(X.data)):
            raise ValueError(f'{name}: expected raw integer counts.')
        native[name] = {'counts': X, 'var': a.var.copy(), 'ids': rows.structure_id.to_numpy(),
                        'position': rows.position.to_numpy(float), 'segment': rows.segment_class.to_numpy(str)}
        join_rows.append({'specimen': name, 'species': species, 'coordinate structures': len(rows),
                          'matrix structures': a.n_obs, 'matched': int((~missing).sum()), 'panel genes': a.n_vars})
    if not native[names[0]]['var'].index.equals(native[names[1]]['var'].index):
        raise ValueError(f'{species}: the two specimens have different panels.')
join = pd.DataFrame(join_rows)
display(join)

# Common support over all four specimens (notebook 37 rule), on the shared axis.
all_pos = np.concatenate([native[n]['position'] for v in SPECIMENS.values() for n in v])
all_spec = np.concatenate([[n] * len(native[n]['position']) for v in SPECIMENS.values() for n in v])
keep_all = common_support_mask(all_pos, all_spec)
offset = 0
for v in SPECIMENS.values():
    for n in v:
        k = len(native[n]['position'])
        native[n]['keep'] = keep_all[offset:offset + k]
        offset += k
lo = max(native[n]['position'][native[n]['keep']].min() for v in SPECIMENS.values() for n in v)
hi = min(native[n]['position'][native[n]['keep']].max() for v in SPECIMENS.values() for n in v)
grid = np.linspace(lo, hi, GRID_POINTS)
knots = gam_internal_knots(all_pos[keep_all], basis_df=BASIS_DF)
for v in SPECIMENS.values():
    for n in v:
        m = native[n]['keep']
        for key in ('counts', 'ids', 'position', 'segment'):
            native[n][key] = native[n][key][m]
        native[n]['library'] = np.asarray(native[n]['counts'].sum(axis=1)).ravel()
        if (native[n]['library'] <= 0).any():
            raise ValueError(f'{n}: structure with zero library.')
support = pd.DataFrame([{'specimen': n, 'structures in common support': len(native[n]['position']),
                         'median library': float(np.median(native[n]['library']))}
                        for v in SPECIMENS.values() for n in v])
display(support)
print(f'Grid {lo:.3f}–{hi:.3f} ({GRID_POINTS} points); knots {np.round(knots, 3).tolist()}')

## 2 · Ortholog classes from HCOP

Orthogroups are connected components of HCOP pairs at a support rule; the primary rule (≥ 3
databases) is the one the paper's ortholog map uses. The paper's map is not strictly
"1:1 orthologs": it keeps the reciprocal best pair (by support count) inside a family. The audit
below shows which family pairs entered the paper's shared space that way.

In [ ]:
hcop = pd.read_csv(HCOP, sep='\t', dtype=str)
panels = {sp: native[names[0]]['var'][['gene_ids']].rename(columns={'gene_ids': 'gene_id'})
          for sp, names in SPECIMENS.items()}
groups_by_rule, classes = {}, {}
for rule in SUPPORT_RULES:
    groups_by_rule[rule] = pz.hcop_orthogroups(hcop, min_support=rule)
    for sp in SPECIMENS:
        classes[(sp, rule)] = pz.classify_panel(panels[sp], sp, groups_by_rule[rule])
class_counts = pd.DataFrame({f'{sp} · support ≥ {rule}': classes[(sp, rule)].ortholog_class.value_counts()
                             for sp in SPECIMENS for rule in SUPPORT_RULES}).reindex(pz.CLASS_ORDER).fillna(0).astype(int)
class_counts.loc['matched by symbol (not id)'] = [int(classes[(sp, r)].matched_by.eq('symbol').sum())
                                                  for sp in SPECIMENS for r in SUPPORT_RULES]
display(class_counts)
class_counts.to_csv(tables / 'panel_ortholog_classes_by_support_rule.csv')

groups = groups_by_rule[PRIMARY_SUPPORT]
paper_map = pd.read_csv(PAPER_MAP)
paper_mouse = dict(zip(paper_map.mouse_symbol, paper_map.human_symbol))
paper_human = dict(zip(paper_map.human_symbol, paper_map.mouse_symbol))
gene_tables = {}
for sp in SPECIMENS:
    t = classes[(sp, PRIMARY_SUPPORT)].copy()
    t['species'] = sp
    t['paper_partner'] = t.index.map(paper_mouse if sp == 'mouse' else paper_human)
    for rule in SUPPORT_RULES:
        t[f'class_support{rule}'] = classes[(sp, rule)].ortholog_class
    gene_tables[sp] = t
# Members of each primary orthogroup, by species (all HCOP genes, including genes outside the panels).
members = groups[groups.component >= 0].groupby(['component', 'species']).symbol.apply(lambda s: sorted(s)).unstack()
in_family = {sp: gene_tables[sp][gene_tables[sp].ortholog_class.isin(['1:many', 'many:1', 'many:many'])] for sp in SPECIMENS}
paper_from_family = paper_map[paper_map.mouse_symbol.isin(in_family['mouse'].index) |
                              paper_map.human_symbol.isin(in_family['human'].index)]
print(f'Paper map: {len(paper_map):,} accepted pairs; {len(paper_from_family):,} of them lie in non-1:1 '
      f'orthogroups at support ≥ {PRIMARY_SUPPORT} (kept by reciprocal best match).')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
display(class_counts)

## 3 · Probe audit: does any probe also match a paralog?

Panels: mouse = public Visium Mouse v2.0 (its included genes equal our matrices); human = the
probes the run actually included (`molecule_info.h5`, panel v2.1.0 on GRCh38-2024-A). Human probe
sequences come from the public v2.0 file, matched by probe hash, because v2.1.0 renamed some genes
but kept their probes. Every included probe is aligned to its species' transcriptome (cDNA + ncRNA)
with bowtie2: end to end, no gaps, up to 5 mismatches, all alignments reported.

In [ ]:
import h5py

from pseudospace.stage_cache import code_digest, stage_is_fresh, stage_mark_fresh

mouse_panel = pd.read_csv(PROBES['mouse'], comment='#')
mouse_panel = mouse_panel[mouse_panel.included.astype(str).str.upper().eq('TRUE')]
if set(mouse_panel.gene_id) != set(panels['mouse'].gene_id):
    raise ValueError('Mouse matrices and the mouse v2.0 panel disagree.')
run_tables = []
for path in MOLECULE_INFO:
    with h5py.File(path, 'r') as handle:
        probes_h5 = handle['probes']
        run_tables.append(pd.DataFrame({'probe_id': probes_h5['probe_id'][:].astype(str),
                                        'gene_id': probes_h5['feature_id'][:].astype(str),
                                        'included': probes_h5['included_probes'][:].astype(bool)}))
        panel_name = json.loads(handle['metrics_json'][()]).get('target_panel_name')
pd.testing.assert_frame_equal(run_tables[0], run_tables[1])
human_run = run_tables[0][run_tables[0].included].copy()
public_human = pd.read_csv(PROBES['human'], comment='#')
public_human['hash'] = public_human.probe_id.str.split('|').str[2]
human_run['hash'] = human_run.probe_id.str.split('|').str[2]
# Match on gene id + hash; fall back to the hash alone where it is unique in the public file
# (one hash is shared by two genes there).
by_gene_hash = public_human.set_index(public_human.gene_id + '|' + public_human.hash).probe_seq
unique_hash = public_human[~public_human.hash.duplicated(keep=False)].set_index('hash').probe_seq
human_run['probe_seq'] = (human_run.gene_id + '|' + human_run.hash).map(by_gene_hash)
human_run['probe_seq'] = human_run.probe_seq.fillna(human_run.hash.map(unique_hash))
if human_run.probe_seq.isna().any():
    raise ValueError(f'{int(human_run.probe_seq.isna().sum())} run probes have no public sequence.')
if set(human_run.gene_id) != set(panels['human'].gene_id):
    raise ValueError('Human matrices and the run panel disagree.')
probe_sets = {'mouse': mouse_panel[['probe_id', 'gene_id', 'probe_seq']].reset_index(drop=True),
              'human': human_run[['probe_id', 'gene_id', 'probe_seq']].reset_index(drop=True)}
print(f'Human run panel: {panel_name}; {len(human_run):,} included probes; '
      f'{int(human_run.probe_id.isin(public_human.probe_id).sum()):,} with identical ids in v2.0')


def align_species(species):
    """Write probes, build the transcriptome index if needed, align, return per-probe status."""
    probes = probe_sets[species]
    fasta = alignment_dir / f'{species}_probes.fa'
    text = ''.join(f'>{p}\n{s}\n' for p, s in zip(probes.probe_id, probes.probe_seq))
    if not fasta.is_file() or fasta.read_text() != text:
        fasta.write_text(text)
    index = alignment_dir / f'{species}_transcriptome'
    index_key = digest({'fasta': [Path(p) for p in TRANSCRIPTS[species]], 'tool': 'bowtie2-build'})
    marker = alignment_dir / f'{species}_transcriptome.index'
    if not stage_is_fresh(marker, index_key):
        combined = alignment_dir / f'{species}_transcripts.fa'
        pz.write_combined_fasta(TRANSCRIPTS[species], combined)
        import subprocess
        subprocess.run(['bowtie2-build', '--threads', str(THREADS), '-q', str(combined), str(index)], check=True)
        combined.unlink()
        marker.write_text(index_key)
        stage_mark_fresh(marker, index_key, stage=f'{species}_index')
    sam = alignment_dir / f'{species}_probes.sam'
    sam_key = digest({'probes': probes, 'index': index_key, 'max_mismatches': NEAR_MM,
                      'code': code_digest(pz.run_bowtie2)})
    if not stage_is_fresh(sam, sam_key):
        pz.run_bowtie2(index, fasta, sam, threads=THREADS, max_mismatches=NEAR_MM)
        stage_mark_fresh(sam, sam_key, stage=f'{species}_alignment')
    transcripts = pz.transcript_table(TRANSCRIPTS[species])
    alignments = pz.read_alignments(sam)
    probe_gene = probes.set_index('probe_id').gene_id
    status = pz.probe_offtarget(alignments, probe_gene, transcripts, cross=CROSS_MM, near=NEAR_MM)
    status_all = pz.probe_offtarget(alignments, probe_gene, transcripts, cross=CROSS_MM, near=NEAR_MM,
                                    count_pseudogenes=True)
    genes_primary = pz.gene_probe_status(status, probe_gene)
    genes_all = pz.gene_probe_status(status_all, probe_gene).add_suffix('_with_pseudogenes')
    return status, genes_primary.join(genes_all), status.attrs['on_target_reverse'], len(alignments)


probe_status, gene_probe = {}, {}
for sp in SPECIMENS:
    probe_status[sp], gene_probe[sp], reverse, n_aln = align_species(sp)
    probe_status[sp].to_csv(tables / f'probe_status_{sp}.csv')
    print(f'{sp}: {n_aln:,} alignments; on-target strand {"reverse" if reverse else "forward"}; '
          f'probe status {probe_status[sp].status.value_counts().to_dict()}')
    t = gene_tables[sp]
    gp = gene_probe[sp].reindex(t.gene_id)
    t['n_probes'] = gp.n_probes.fillna(0).astype(int).to_numpy()
    t['probe_status'] = gp.probe_status.fillna('absent').to_numpy()
    t['best_offtarget_mm'] = gp.best_offtarget_mm.to_numpy()
    t['offtarget_genes'] = gp.offtarget_genes.fillna('').to_numpy()
    t['probe_status_with_pseudogenes'] = gp.probe_status_with_pseudogenes.fillna('absent').to_numpy()
    t['probe_clean'] = t.probe_status.isin(['clean', 'near'])
    t['probe_clean_strict'] = t.probe_status.eq('clean')
probe_summary = pd.DataFrame({sp: gene_tables[sp].groupby('ortholog_class').probe_status.value_counts().unstack(fill_value=0).stack()
                              for sp in SPECIMENS}).fillna(0).astype(int)
display(probe_summary.unstack(level=1, fill_value=0))

## 4 · Stage 1 · PT expression and zonation within each species

Each specimen gets its own library-weighted spline per gene on the common grid; the weighted F-test
compares it with a flat line. A species-level F-test with specimen offsets is reported beside the
per-specimen calls but does not enter them. No ortholog mapping is involved.

In [ ]:
SEGMENTS = ('PT-S1', 'PT-S2', 'PT-S3')
grid_design = make_gam_design(grid, knots)


def specimen_fits(name):
    d = native[name]
    X, L = d['counts'], d['library']
    E = sparse.diags(1e4 / L) @ X
    w = L / L.mean()
    B = make_gam_design(d['position'], knots)
    coef, F, p, df1, df2 = pz.weighted_nested_f(E, w, B, np.ones((len(L), 1)))
    seg = np.stack([np.asarray(X[d['segment'] == s].sum(axis=0)).ravel() / L[d['segment'] == s].sum() * 1e4
                    for s in SEGMENTS], axis=1)
    return {'curve': (grid_design @ coef).T.astype(np.float32), 'F': F, 'p': p,
            'detection': np.asarray((X > 0).mean(axis=0)).ravel(),
            'pseudobulk': np.asarray(X.sum(axis=0)).ravel() / L.sum() * 1e4, 'segment_means': seg,
            'df': np.array([df1, df2])}


def species_pooled_test(species):
    names = SPECIMENS[species]
    E = sparse.vstack([sparse.diags(1e4 / native[n]['library']) @ native[n]['counts'] for n in names]).tocsr()
    L = np.concatenate([native[n]['library'] for n in names])
    pos = np.concatenate([native[n]['position'] for n in names])
    offsets = np.zeros((len(L), len(names)))
    start_row = 0
    for k, n in enumerate(names):
        offsets[start_row:start_row + len(native[n]['library']), k] = 1
        start_row += len(native[n]['library'])
    full = np.column_stack([offsets, make_gam_design(pos, knots)[:, 1:]])
    _, F, p, *_ = pz.weighted_nested_f(E, L / L.mean(), full, offsets)
    return {'F': F, 'p': p}


fits = {}
for sp, names in SPECIMENS.items():
    for n in names:
        fits[n] = cached_payload(
            f'fit_{n}', lambda n=n: specimen_fits(n), root=cache_root,
            params={'logic': NOTEBOOK_LOGIC_VERSION, 'knots': knots, 'grid': grid},
            inputs={'counts': native[n]['counts'], 'position': native[n]['position'], 'segment': native[n]['segment']},
            code=[specimen_fits, pz.weighted_nested_f, make_gam_design])
    fits[sp] = cached_payload(
        f'pooled_{sp}', lambda sp=sp: species_pooled_test(sp), root=cache_root,
        params={'logic': NOTEBOOK_LOGIC_VERSION, 'knots': knots},
        inputs={n: native[n]['counts'] for n in names} | {f'{n}_pos': native[n]['position'] for n in names},
        code=[species_pooled_test, pz.weighted_nested_f])


def zonation_table(species, pseudocount):
    names = SPECIMENS[species]
    t = gene_tables[species]
    a, b = (fits[n] for n in names)
    out = pd.DataFrame(index=t.index)
    for tag, f in zip(('1', '2'), (a, b)):
        out[f'detection_{tag}'] = f['detection']
        out[f'cp10k_{tag}'] = f['pseudobulk']
    out['pt_expressed'] = out[['detection_1', 'detection_2']].min(axis=1).ge(DETECTION)
    phi = {tag: pz.log_profile(f['curve'], pseudocount) for tag, f in zip(('1', '2'), (a, b))}
    for tag, f in zip(('1', '2'), (a, b)):
        q = np.full(len(out), np.nan)
        q[out.pt_expressed.to_numpy()] = bh_adjust(f['p'][out.pt_expressed.to_numpy()])
        out[f'q_{tag}'] = q
        out[f'amplitude_{tag}'] = phi[tag].max(axis=1) - phi[tag].min(axis=1)
        out[f'peak_{tag}'] = grid[np.argmax(phi[tag], axis=1)]
    out['replication_r'] = pz.rowwise_pearson(phi['1'], phi['2'])
    pooled_phi = (phi['1'] + phi['2']) / 2
    out['peak'] = grid[np.argmax(pooled_phi, axis=1)]
    out['trough'] = grid[np.argmin(pooled_phi, axis=1)]
    out['log2_cp10k'] = np.log2((out.cp10k_1 + out.cp10k_2) / 2 + pseudocount)
    out['pooled_species_p'] = fits[species]['p']
    out['zonated'] = (out.pt_expressed & out.q_1.lt(Q_MAX) & out.q_2.lt(Q_MAX)
                      & out.amplitude_1.ge(MIN_AMPLITUDE) & out.amplitude_2.ge(MIN_AMPLITUDE)
                      & out.replication_r.ge(MIN_REPLICATION))
    out['Z'] = np.where(out.zonated, out[['amplitude_1', 'amplitude_2']].min(axis=1), 0.)
    out['strong'] = out.Z.ge(STRONG)
    seg = (a['segment_means'] + b['segment_means']) / 2
    for k, s in enumerate(SEGMENTS):
        out[f'segment_cp10k_{s[-2:]}'] = seg[:, k]
    out['peak_segment'] = np.array([s[-2:] for s in SEGMENTS])[np.argmax(seg, axis=1)]
    for tag, f in zip(('1', '2'), (a, b)):
        out[f'peak_segment_{tag}'] = np.array([s[-2:] for s in SEGMENTS])[np.argmax(f['segment_means'], axis=1)]
    return out, phi


zon, phis, zon_sens = {}, {}, {}
for sp in SPECIMENS:
    zon[sp], phis[sp] = zonation_table(sp, PSEUDOCOUNT)
    zon_sens[sp], _ = zonation_table(sp, PSEUDOCOUNT_SENSITIVITY)
    gene_tables[sp] = gene_tables[sp].drop(columns=[c for c in zon[sp] if c in gene_tables[sp]]).join(zon[sp])
    gene_tables[sp]['zonated_pc02'] = zon_sens[sp].zonated
    gene_tables[sp]['strong_pc02'] = zon_sens[sp].strong

# Construction genes (notebook 36 payload) and their human partners in the paper's map.
import glob
payload_files = glob.glob(str(NB36_CACHE / 'arm_scf13_full__*.npz'))
if len(payload_files) != 1:
    raise FileNotFoundError('Notebook 36 payload arm_scf13_full not found (construction genes).')
construction_mouse = set(np.load(payload_files[0], allow_pickle=True)['construction'].astype(str))
construction_human = {paper_human_partner for m, paper_human_partner in paper_mouse.items() if m in construction_mouse}
gene_tables['mouse']['construction_gene'] = gene_tables['mouse'].index.isin(sorted(construction_mouse))
gene_tables['human']['construction_gene'] = gene_tables['human'].index.isin(sorted(construction_human))
gene_tables['mouse']['flag_fasting_ppara'] = gene_tables['mouse'].index.isin(sorted(FASTING_PPARA))
gene_tables['mouse']['flag_androgen'] = gene_tables['mouse'].index.isin(sorted(ANDROGEN))
gene_tables['human']['flag_fasting_ppara'] = gene_tables['human'].index.isin([g.upper() for g in FASTING_PPARA])
gene_tables['human']['flag_androgen'] = False
for sp in SPECIMENS:
    gene_tables[sp].to_csv(tables / f'gene_inventory_{sp}.csv')
print({sp: f"{len(construction_mouse if sp == 'mouse' else construction_human)} construction genes" for sp in SPECIMENS})

### 4a · Inventory: how many genes of each class are measured, PT-expressed and zonated?

Counts below use the primary support rule (≥ 3 databases) and probe-clean genes; ambiguous genes are
counted separately and never enter a claim.

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import StratifiedTable

FAMILY = ['1:many', 'many:1', 'many:many']


def inventory_counts(species, class_col='ortholog_class'):
    t = gene_tables[species]
    rows = []
    for cls in pz.CLASS_ORDER:
        s = t[t[class_col].eq(cls)]
        amb = s.probe_status.eq('ambiguous')
        rows.append({'species': species, 'class': cls, 'in panel': len(s), 'probe-clean': int(s.probe_clean.sum()),
                     'probe-ambiguous': int(amb.sum()), 'unverified': int(s.probe_status.eq('unverified').sum()),
                     'PT-expressed, clean': int((s.pt_expressed & s.probe_clean).sum()),
                     'zonated, clean': int((s.zonated & s.probe_clean).sum()),
                     'strong, clean': int((s.strong & s.probe_clean).sum()),
                     'PT-expressed, ambiguous': int((s.pt_expressed & amb).sum()),
                     'zonated, ambiguous': int((s.zonated & amb).sum())})
    return pd.DataFrame(rows)


inventory = pd.concat([inventory_counts(sp) for sp in SPECIMENS], ignore_index=True)
display(inventory)
inventory.to_csv(tables / 'inventory_counts.csv', index=False)


def share_statistic(species, label, *, class_col='ortholog_class', clean_col='probe_clean', strong_col='strong',
                    exclude_construction=False, outcome='strong'):
    t = gene_tables[species]
    base = t[t.pt_expressed & t[clean_col]]
    if exclude_construction:
        base = base[~base.construction_gene]
    non = ~base[class_col].eq('1:1')
    hit = base[strong_col] if outcome == 'strong' else base[strong_col.replace('strong', 'zonated')]
    table = [[int((hit & non).sum()), int((hit & ~non).sum())], [int((~hit & non).sum()), int((~hit & ~non).sum())]]
    odds, p = fisher_exact(table)
    strata = pd.qcut(base.log2_cp10k, 5, labels=False)
    mh = StratifiedTable([np.array([[((hit & non) & (strata == k)).sum(), ((hit & ~non) & (strata == k)).sum()],
                                    [((~hit & non) & (strata == k)).sum(), ((~hit & ~non) & (strata == k)).sum()]])
                          + .5 for k in range(5)])
    z_all = base.Z if outcome == 'strong' else base.Z
    return {'species': species, 'variant': label, 'PT-expressed clean genes': len(base),
            'non-1:1 share of PT-expressed': non.mean(), f'{outcome} genes': int(hit.sum()),
            f'non-1:1 share of {outcome}': (hit & non).sum() / max(hit.sum(), 1),
            'non-1:1 share of Z mass': z_all[non].sum() / z_all.sum(),
            'odds ratio': odds, 'Fisher p': p, 'MH odds ratio (expression quintiles)': mh.oddsratio_pooled,
            'MH p': mh.test_null_odds().pvalue}


variants = []
for sp in SPECIMENS:
    variants += [share_statistic(sp, 'primary'),
                 share_statistic(sp, 'without construction genes', exclude_construction=True),
                 share_statistic(sp, 'pseudocount 0.2', strong_col='strong_pc02'),
                 share_statistic(sp, 'support ≥ 2', class_col='class_support2'),
                 share_statistic(sp, 'support ≥ 5', class_col='class_support5'),
                 share_statistic(sp, 'strict probe rule', clean_col='probe_clean_strict')]
share = pd.DataFrame(variants)
display(share.round(3))
share.to_csv(tables / 'nonone_share_of_strong_zonation.csv', index=False)

by_class = []
for sp in SPECIMENS:
    t = gene_tables[sp]
    base = t[t.pt_expressed & t.probe_clean]
    for cls in pz.CLASS_ORDER:
        s = base[base.ortholog_class.eq(cls)]
        by_class.append({'species': sp, 'class': cls, 'PT-expressed clean': len(s),
                         'share of PT-expressed': len(s) / len(base),
                         'zonated': int(s.zonated.sum()), 'share of zonated': s.zonated.sum() / base.zonated.sum(),
                         'strong': int(s.strong.sum()), 'share of strong': s.strong.sum() / base.strong.sum(),
                         'zonated fraction within class': s.zonated.mean() if len(s) else np.nan})
by_class = pd.DataFrame(by_class)
display(by_class.round(3))
by_class.to_csv(tables / 'zonation_by_class.csv', index=False)

**Reading.** In mouse, genes outside the 1:1 space are about twice as likely as 1:1 genes to be
strongly zonated, in every variant (support rule, pseudocount, probe rule, construction genes removed,
expression-matched). In human they are not. Most of the mouse excess sits in many:many and 1:many
families: 27% of PT-expressed 1:many genes and 30% of many:many genes are zonated, against 18% of 1:1
genes. Genes without any ortholog are rarely PT-expressed in mouse (11 genes) and contribute little.

### 4b · Catalogue of non-1:1 families and the paper map's picks inside them

In [ ]:
def describe(t, rows):
    return '; '.join(f'{g} ({r.peak:.2f}, Z {r.Z:.1f})' for g, r in t.loc[rows].sort_values('peak').iterrows())


catalogue = []
family_components = sorted(set(gene_tables['mouse'].loc[gene_tables['mouse'].ortholog_class.isin(FAMILY), 'component'])
                           | set(gene_tables['human'].loc[gene_tables['human'].ortholog_class.isin(FAMILY), 'component']))
paper_pairs = paper_map.assign(mouse_component=paper_map.mouse_symbol.map(gene_tables['mouse'].component),
                               human_component=paper_map.human_symbol.map(gene_tables['human'].component))
for comp in family_components:
    row = {'component': comp}
    for sp in SPECIMENS:
        t = gene_tables[sp]
        s = t[t.component.eq(comp)]
        clean_expr = s[s.pt_expressed & s.probe_clean]
        zon_clean = s[s.zonated & s.probe_clean]
        row[f'{sp} class'] = s.ortholog_class.iloc[0] if len(s) else ''
        row[f'{sp} members in panel'] = ', '.join(sorted(s.index))
        row[f'{sp} n panel'] = len(s)
        row[f'{sp} n PT-expressed clean'] = len(clean_expr)
        row[f'{sp} n zonated clean'] = len(zon_clean)
        row[f'{sp} zonated clean (peak, Z)'] = describe(t, zon_clean.index) if len(zon_clean) else ''
        row[f'{sp} PT-expressed ambiguous'] = ', '.join(sorted(s.index[s.pt_expressed & s.probe_status.eq('ambiguous')]))
        row[f'{sp} top PT member'] = (s.loc[s.pt_expressed].log2_cp10k.idxmax() if s.pt_expressed.any() else '')
        row[f'{sp} total HCOP members'] = len(members.loc[comp, sp]) if comp in members.index and isinstance(members.loc[comp, sp], list) else 0
    kept = paper_pairs[paper_pairs.mouse_component.eq(comp) | paper_pairs.human_component.eq(comp)]
    row['paper map pairs'] = '; '.join(f'{h}–{m}' for h, m in zip(kept.human_symbol, kept.mouse_symbol))
    row['paper mouse partner PT-expressed'] = bool(gene_tables['mouse'].reindex(kept.mouse_symbol).pt_expressed.eq(True).any()) if len(kept) else None
    row['paper partner is top mouse PT member'] = (row['mouse top PT member'] in set(kept.mouse_symbol)) if len(kept) and row['mouse top PT member'] else None
    row['paper partner is top human PT member'] = (row['human top PT member'] in set(kept.human_symbol)) if len(kept) and row['human top PT member'] else None
    row['zonated clean in both species'] = row['mouse n zonated clean'] > 0 and row['human n zonated clean'] > 0
    row['PT-expressed clean in both species'] = row['mouse n PT-expressed clean'] > 0 and row['human n PT-expressed clean'] > 0
    row['≥2 zonated clean paralogs in'] = ', '.join(sp for sp in SPECIMENS if row[f'{sp} n zonated clean'] >= 2)
    flags = set(gene_tables['mouse'].index[gene_tables['mouse'].component.eq(comp) & gene_tables['mouse'].flag_fasting_ppara]) | \
        set(gene_tables['mouse'].index[gene_tables['mouse'].component.eq(comp) & gene_tables['mouse'].flag_androgen])
    row['state flags (mouse)'] = ', '.join(sorted(flags))
    catalogue.append(row)
catalogue = pd.DataFrame(catalogue)
catalogue.to_csv(tables / 'family_catalogue.csv', index=False)
expressed_families = catalogue[catalogue['mouse n PT-expressed clean'].gt(0) | catalogue['human n PT-expressed clean'].gt(0)]
print(f'{len(catalogue):,} non-1:1 orthogroups with a panel gene; {len(expressed_families):,} with a PT-expressed '
      f'probe-clean member in either species; {int(catalogue["PT-expressed clean in both species"].sum())} in both.')
audit_rows = []
for sp in SPECIMENS:
    with_member = catalogue[catalogue[f'{sp} n PT-expressed clean'].gt(0) & catalogue['paper map pairs'].ne('')]
    t = gene_tables[sp]
    kept_expressed = [bool(t[t.component.eq(c) & t.paper_partner.notna()].pt_expressed.any()) for c in with_member.component]
    audit_rows.append({'species': sp, 'families with a PT-expressed clean member and a kept pair': len(with_member),
                       'kept partner PT-expressed': int(np.sum(kept_expressed)),
                       'kept partner is the most expressed PT member': int(with_member[f'paper partner is top {sp} PT member'].eq(True).sum())})
paper_audit = pd.DataFrame(audit_rows)
display(paper_audit)
paper_audit.to_csv(tables / 'paper_map_audit.csv', index=False)
show = ['component', 'human members in panel', 'mouse members in panel', 'human zonated clean (peak, Z)',
        'mouse zonated clean (peak, Z)', 'human PT-expressed ambiguous', 'mouse PT-expressed ambiguous', 'paper map pairs',
        'state flags (mouse)']
display(catalogue[catalogue['zonated clean in both species'] | catalogue['≥2 zonated clean paralogs in'].ne('')][show])

### 4b′ · The strongly zonated non-1:1 genes and their families in the other species (descriptive)

For each strongly zonated, probe-clean non-1:1 gene: is it in the paper's shared space (kept by
reciprocal best match), and what do the members of its orthogroup do in the other species? Human
GSTT1 lies on an alternate GRCh38 scaffold and is not on the human panel; mouse Gsta4 and human GSTA4
are absent from this HCOP copy (and from Compara 98) although the symbols match, so their
`no_ortholog` class is an annotation gap, flagged below.

In [ ]:
def counterpart_status(other, comp):
    if comp < 0:
        return 'no HCOP ortholog'
    to = gene_tables[other]
    s = to[to.component.eq(comp)]
    if s.empty:
        return 'no member on the other panel'
    if (s.zonated & s.probe_clean).any():
        return 'zonated in the other species'
    if (s.pt_expressed & s.probe_clean).any():
        return 'PT-expressed, not zonated'
    if s.pt_expressed.any():
        return 'PT-expressed, probe-ambiguous only'
    return 'not PT-expressed'


strong_rows = []
for sp, other in (('mouse', 'human'), ('human', 'mouse')):
    t = gene_tables[sp]
    other_upper = set(gene_tables[other].index.str.upper())
    for g, r in t[t.strong & t.probe_clean & ~t.ortholog_class.eq('1:1')].sort_values('Z', ascending=False).iterrows():
        status = counterpart_status(other, r.component)
        if r.ortholog_class == 'no_ortholog' and g.upper() in other_upper:
            status = 'annotation gap: same symbol on the other panel'
        strong_rows.append({'species': sp, 'gene': g, 'class': r.ortholog_class, 'Z': round(r.Z, 2), 'peak': round(r.peak, 2),
                            'peak segment': r.peak_segment, 'in paper shared space': isinstance(r.paper_partner, str),
                            'other species': status, 'family root': pz.symbol_root(g),
                            'flags': ', '.join(f for f, on in (('fasting/PPARα', bool(r.flag_fasting_ppara)),
                                                               ('androgen', bool(r.flag_androgen))) if on)})
strong_non11 = pd.DataFrame(strong_rows)
strong_non11.to_csv(tables / 'strong_non_one_to_one_genes.csv', index=False)
display(strong_non11.groupby(['species', 'other species']).size().unstack(fill_value=0))
display(strong_non11.groupby('species')['in paper shared space'].agg(['sum', 'size']))
display(strong_non11[strong_non11.species.eq('mouse')].head(40))
display(strong_non11[strong_non11.species.eq('human')])

### 4c · Stop rule

In [ ]:
F_both = int(catalogue['zonated clean in both species'].sum())
F_partition = int(catalogue['≥2 zonated clean paralogs in'].ne('').sum())
specific = {sp: gene_tables[sp][gene_tables[sp].ortholog_class.eq('no_ortholog') & gene_tables[sp].zonated
                                & gene_tables[sp].probe_clean] for sp in SPECIMENS}
n_specific = max(len(v) for v in specific.values())
STAGE2 = {'a_c': F_both >= STOP_FAMILIES, 'b_test': F_partition >= MIN_PARTITION_FAMILIES,
          'd': n_specific >= MIN_SPECIFIC}
print(f'Families with a zonated, probe-clean member in each species: {F_both} (stop if < {STOP_FAMILIES}).')
print(f'Families with ≥ 2 zonated, probe-clean paralogs in one species: {F_partition} (test needs ≥ {MIN_PARTITION_FAMILIES}).')
print(f'Zonated, probe-clean genes without an ortholog: ' + ', '.join(f'{sp} {len(v)}' for sp, v in specific.items()))
print('Stage 2 decisions:', STAGE2)

**Decision.** The pre-declared stop rule does not fire for stages 2(a)–(c): 20 families have a zonated,
probe-clean member in each species (threshold 10), and 22 have at least two zonated, probe-clean paralogs
in one species (threshold 8). Stage 2(d) does not run: at most 9 zonated, probe-clean genes without any
ortholog exist in a species (threshold 10). The mouse list includes Gsta4, a 1:1 gene missing from this
HCOP copy, and Gstt1, whose human ortholog lies on an alternate GRCh38 scaffold and is not on the panel.

### 4d · Figure 1 · Inventory summary

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 8.5,
                     'axes.titlesize': 9, 'axes.titleweight': 'bold', 'pdf.fonttype': 42, 'svg.fonttype': 'none'})
HUMAN, MOUSE, INK, MUTED, GRID = '#D55E00', '#0072B2', '#333333', '#8C8C8C', '#E6E6E6'
SPECIES_COLOR = {'mouse': MOUSE, 'human': HUMAN}
CLASS_LABEL = {'1:1': '1:1', '1:many': '1:many', 'many:1': 'many:1', 'many:many': 'many:many',
               'no_consensus': 'low support only', 'no_ortholog': 'no ortholog'}


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=200)


fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), gridspec_kw={'width_ratios': [1.35, 1, 1.1], 'wspace': .45})
ax = axes[0]
classes_shown = [c for c in pz.CLASS_ORDER if c != '1:1']
ypos = np.arange(len(classes_shown))
for k, sp in enumerate(SPECIMENS):
    inv = inventory[inventory.species.eq(sp)].set_index('class').loc[classes_shown]
    offset = (-.2 if sp == 'mouse' else .2)
    for col, alpha in (('PT-expressed, clean', .25), ('zonated, clean', .6), ('strong, clean', 1.)):
        ax.barh(ypos + offset, inv[col], height=.36, color=SPECIES_COLOR[sp], alpha=alpha, lw=0)
    for y, v in zip(ypos + offset, inv['zonated, clean']):
        ax.text(v, y, f' {v}', va='center', fontsize=6.5, color=INK)
ax.set_yticks(ypos, [CLASS_LABEL[c] for c in classes_shown])
ax.invert_yaxis()
ax.set_xlabel('genes (probe-clean)')
ax.set_title('Non-1:1 genes measured in PT', loc='left')
ax.legend(handles=[Patch(color=MUTED, alpha=a, label=l) for a, l in ((.25, 'PT-expressed'), (.6, 'zonated'), (1., 'strongly zonated'))]
          + [Patch(color=MOUSE, label='mouse'), Patch(color=HUMAN, label='human')], fontsize=6.5, frameon=False,
          loc='upper center', bbox_to_anchor=(.5, -.18), ncol=5)

ax = axes[1]
levels = ['PT-expressed', 'zonated', 'strong']
for sp in SPECIMENS:
    t = gene_tables[sp]
    base = t[t.pt_expressed & t.probe_clean]
    non = ~base.ortholog_class.eq('1:1')
    values = [non.mean(), non[base.zonated].mean(), non[base.strong].mean()]
    ax.plot(range(3), values, '-o', color=SPECIES_COLOR[sp], lw=2, ms=6, label=sp)
    for x, v in enumerate(values):
        ax.text(x + .08, v, f'{100 * v:.0f}%', fontsize=6.5, color=INK, va='center')
ax.set_xticks(range(3), ['PT-expressed', 'zonated', 'strongly\nzonated'])
ax.set_ylabel('share that is non-1:1')
ax.set_ylim(0, None)
ax.grid(axis='y', color=GRID, lw=.6)
ax.set_title('Share of zonation in non-1:1 genes', loc='left')
ax.legend(frameon=False, fontsize=7)

ax = axes[2]
for k, sp in enumerate(SPECIMENS):
    t = gene_tables[sp]
    z = t[t.zonated & t.probe_clean]
    groups_z = [z.Z[z.ortholog_class.eq('1:1')], z.Z[z.ortholog_class.isin(FAMILY)],
                z.Z[z.ortholog_class.isin(['no_consensus', 'no_ortholog'])]]
    pos = np.arange(3) * 2.6 + k * 1.0
    parts = ax.boxplot(groups_z, positions=pos, widths=.8, patch_artist=True, showfliers=False,
                       medianprops={'color': INK, 'lw': 1.2}, whiskerprops={'color': MUTED}, capprops={'color': MUTED})
    for b in parts['boxes']:
        b.set(facecolor=SPECIES_COLOR[sp], alpha=.35, edgecolor=SPECIES_COLOR[sp])
    for p_, g in zip(pos, groups_z):
        ax.text(p_, -.08, f'n={len(g)}', ha='center', va='top', fontsize=6, color=MUTED, transform=ax.get_xaxis_transform())
ax.set_xticks(np.arange(3) * 2.6 + .5, ['\n1:1', '\nfamilies', '\nno / low-support\northolog'])
ax.set_ylabel('zonation strength Z (log2, min over specimens)')
ax.set_title('Strength of zonated genes by class', loc='left')
ax.grid(axis='y', color=GRID, lw=.6)
save(fig, 'fig1_inventory')
plt.show()
plt.close(fig)

## 5 · Stage 2(a) · Family units compared across species

Each eligible family (a zonated, probe-clean member in each species) becomes one unit per specimen:
the sum of its PT-expressed, probe-clean members' curves (splines are linear in the data, so this is
the curve of the summed expression). A family is called different in shape between species when the
two within-species distances are the two smallest of the six specimen pairs and the cross-species
mean exceeds the within-species mean by ≥ 0.5 (RMS log2 of mean-centred profiles). The same rule on
the two balanced relabelings is the specificity control; 1:1 genes are the comparator.

In [ ]:
from pseudospace.pathway_calibration import balanced_partitions
from scipy.stats import wilcoxon

ALL_SPECIMENS = SPECIMENS['mouse'] + SPECIMENS['human']
partitions = balanced_partitions(ALL_SPECIMENS, ['mouse', 'mouse', 'human', 'human'])
partition_groups = {label: ['b' if n in members_ else 'a' for n in ALL_SPECIMENS]
                    for label, members_ in partitions.items()}
row_of = {sp: pd.Series(np.arange(len(gene_tables[sp])), index=gene_tables[sp].index) for sp in SPECIMENS}


def unit_curves(species, genes_):
    """Linear-scale summed curves of ``genes_`` for each specimen of ``species`` (2 x grid)."""
    idx = row_of[species].loc[list(genes_)].to_numpy()
    return np.stack([fits[n]['curve'][idx].sum(axis=0) for n in SPECIMENS[species]])


def unit_phi(curves):
    return pz.log_profile(curves, PSEUDOCOUNT)


def unit_row(label, mouse_genes, human_genes):
    cm, ch = unit_curves('mouse', mouse_genes), unit_curves('human', human_genes)
    P = np.vstack([unit_phi(cm), unit_phi(ch)])
    row = {'unit': label, 'mouse genes': ', '.join(mouse_genes), 'human genes': ', '.join(human_genes)}
    for part, groups_ in partition_groups.items():
        call, gap, within, between = pz.separation_call(P, groups_, min_gap=SEPARATION_GAP)
        row[f'call {part}'], row[f'gap {part}'] = call, gap
    row['cross-species r'] = float(np.mean([pz.rowwise_pearson(P[i][None], P[j][None])[0]
                                            for i in (0, 1) for j in (2, 3)]))
    row['within r mouse'] = float(pz.rowwise_pearson(P[0][None], P[1][None])[0])
    row['within r human'] = float(pz.rowwise_pearson(P[2][None], P[3][None])[0])
    for sp, P2 in (('mouse', P[:2]), ('human', P[2:])):
        amp = P2.max(axis=1) - P2.min(axis=1)
        row[f'{sp} Z'] = float(amp.min())
        row[f'{sp} peak'] = float(grid[np.argmax(P2.mean(axis=0))])
        row[f'{sp} log2 cp10k'] = float(np.log2(np.mean([fits[n]['pseudobulk'][row_of[sp].loc[list(
            mouse_genes if sp == 'mouse' else human_genes)].to_numpy()].sum() for n in SPECIMENS[sp]]) + PSEUDOCOUNT))
        seg = np.mean([fits[n]['segment_means'][row_of[sp].loc[list(mouse_genes if sp == 'mouse' else human_genes)]
                       .to_numpy()].sum(axis=0) for n in SPECIMENS[sp]], axis=0)
        row[f'{sp} peak segment'] = ['S1', 'S2', 'S3'][int(np.argmax(seg))]
    return row


def clean_expressed(species, comp):
    t = gene_tables[species]
    return sorted(t.index[t.component.eq(comp) & t.pt_expressed & t.probe_clean])


family_units = pd.DataFrame([
    unit_row(f'family {int(r.component)}', clean_expressed('mouse', r.component), clean_expressed('human', r.component))
    for r in catalogue[catalogue['zonated clean in both species']].itertuples()])
family_units['component'] = catalogue.loc[catalogue['zonated clean in both species'], 'component'].to_numpy()

# 1:1 comparator: PT-expressed and probe-clean in both species, zonated in at least one.
one = gene_tables['mouse'][gene_tables['mouse'].ortholog_class.eq('1:1')]
human_by_comp = gene_tables['human'][gene_tables['human'].ortholog_class.eq('1:1')].reset_index().set_index('component')
pairs_11 = []
for m, r in one.iterrows():
    if r.component not in human_by_comp.index:
        continue
    h = human_by_comp.loc[r.component, 'gene']
    hr = gene_tables['human'].loc[h]
    if r.pt_expressed and r.probe_clean and hr.pt_expressed and hr.probe_clean and (r.zonated or hr.zonated):
        pairs_11.append((m, h))
one_units = pd.DataFrame([unit_row(f'{m}/{h}', [m], [h]) for m, h in pairs_11])
for frame in (family_units, one_units):
    frame['max Z'] = frame[['mouse Z', 'human Z']].max(axis=1)
    frame['mean log2 cp10k'] = frame[['mouse log2 cp10k', 'human log2 cp10k']].mean(axis=1)
edges_e = np.quantile(one_units['mean log2 cp10k'], [.2, .4, .6, .8])
edges_z = np.quantile(one_units['max Z'], [.2, .4, .6, .8])
for frame in (family_units, one_units):
    frame['stratum'] = np.searchsorted(edges_e, frame['mean log2 cp10k']) * 5 + np.searchsorted(edges_z, frame['max Z'])

swap_labels = [k for k in partitions if k != 'species']
summary_a = []
for name, frame in (('families', family_units), ('1:1 genes', one_units)):
    species_calls = int(frame['call species'].sum())
    relabeled = [int(frame[f'call {k}'].sum()) for k in swap_labels]
    summary_a.append({'units': name, 'n': len(frame), 'species calls': species_calls,
                      'species call rate': species_calls / len(frame),
                      'relabeled calls': relabeled, 'NCDR': np.mean(relabeled) / max(species_calls, 1),
                      'peak segment differs': int(frame['mouse peak segment'].ne(frame['human peak segment']).sum())})
# Stratum-standardised expectation for families from the 1:1 call rate.
rate_11 = one_units.groupby('stratum')['call species'].mean()
expected = family_units.stratum.map(rate_11)
summary_a = pd.DataFrame(summary_a)
cmh_tables = []
for s, grp in family_units.groupby('stratum'):
    ref = one_units[one_units.stratum.eq(s)]
    if len(ref):
        cmh_tables.append(np.array([[grp['call species'].sum(), (~grp['call species']).sum()],
                                    [ref['call species'].sum(), (~ref['call species']).sum()]], float) + .5)
cmh = StratifiedTable(cmh_tables)
print(f'Families: {int(family_units["call species"].sum())} of {len(family_units)} called; expected from matched 1:1 '
      f'genes {expected.sum():.1f} (stratum-standardised); MH odds ratio {cmh.oddsratio_pooled:.2f}, '
      f'p = {cmh.test_null_odds().pvalue:.3g}; units without a matched stratum: {int(expected.isna().sum())}.')
display(summary_a)
family_units.to_csv(tables / 'stage2a_family_units.csv', index=False)
one_units.to_csv(tables / 'stage2a_one_to_one_units.csv', index=False)
summary_a.to_csv(tables / 'stage2a_summary.csv', index=False)
display(family_units.sort_values('gap species', ascending=False)[
    ['unit', 'mouse genes', 'human genes', 'call species', 'gap species', 'cross-species r', 'within r mouse',
     'within r human', 'mouse peak', 'human peak', 'mouse peak segment', 'human peak segment']].head(25))

**Reading.** Family units differ in shape between species no more often than matched 1:1 genes, and
relabeling calls nothing, so the comparison is specific but not enriched. Of the 6 called families,
Ugt3a1/Ugt3a2 repeats the paper's known Ugt3a1 reversal. Slc22a19/Slc22a26 against SLC22A24 differ in
amplitude, not position: all three peak late (0.78–0.79). The Ugt2b family unit peaks in S2 in mouse
(Ugt2b38, Ugt2b37) and in S1 in human (UGT2B7). Peak segments differ in 12 of 20 families, but segment
argmax is unstable for weakly zonated units.

## 6 · Stage 2(b) · Do paralogs partition the PT axis?

Units are (family, species) with ≥ 2 PT-expressed, zonated, probe-clean paralogs. Each paralog pair is
compared with 1,000 random pairs of zonated, probe-clean genes of the same species, matched member by
member on quintiles of expression and Z and drawn outside the family. Two statistics: distance between
pooled peaks, and shape dissimilarity 1 − r. Both are tested two-sided, because co-regulated tandem
duplicates predict *less* separation and positional subfunctionalisation predicts *more*.

In [ ]:
def partition_analysis(clean_col, seed):
    """Paralog pairs against matched random pairs, for one probe rule. Returns (pairs, unit scores, tests, null)."""
    rng = np.random.default_rng(seed)
    pair_rows, null_store = [], {}
    for sp in SPECIMENS:
        t = gene_tables[sp]
        elig = (t.zonated & t[clean_col]).to_numpy()
        strata = np.full(len(t), -1)
        strata[elig] = (pd.qcut(t.log2_cp10k[elig], 5, labels=False) * 5 + pd.qcut(t.Z[elig], 5, labels=False)).to_numpy()
        fam = np.where(t.ortholog_class.isin(FAMILY), t.component, -1)
        standardized = pz.standardize_rows((phis[sp]['1'] + phis[sp]['2']) / 2)
        peaks, p1, p2 = t.peak.to_numpy(), t.peak_1.to_numpy(), t.peak_2.to_numpy()
        for comp, grp in t[elig & (fam >= 0)].groupby('component'):
            if len(grp) < 2:
                continue
            idx = row_of[sp].loc[grp.sort_values('peak').index].to_numpy()
            pairs = np.array([(i, j) for k, i in enumerate(idx) for j in idx[k + 1:]])
            a, b = pz.matched_pair_draws(pairs, strata, fam, elig, N_DRAWS, rng)
            sep_obs, dis_obs = pz.pair_statistics(peaks, standardized, pairs[:, 0], pairs[:, 1])
            sep_null, dis_null = pz.pair_statistics(peaks, standardized, a, b)
            rep_null = ((np.abs(p1[a] - p1[b]) >= PEAK_SEPARATION) & (np.abs(p2[a] - p2[b]) >= PEAK_SEPARATION)
                        & (np.sign(p1[a] - p1[b]) == np.sign(p2[a] - p2[b])))
            unit = f'{sp}:{comp}'
            null_store[unit] = (sep_null.mean(axis=0), dis_null.mean(axis=0), rep_null.mean(axis=0))
            for k, (i, j) in enumerate(pairs):
                separated = (abs(p1[i] - p1[j]) >= PEAK_SEPARATION and abs(p2[i] - p2[j]) >= PEAK_SEPARATION
                             and np.sign(p1[i] - p1[j]) == np.sign(p2[i] - p2[j]))
                seg_diff = (t.peak_segment_1.iloc[i] != t.peak_segment_1.iloc[j]
                            and t.peak_segment_2.iloc[i] != t.peak_segment_2.iloc[j])
                pair_rows.append({'unit': unit, 'species': sp, 'component': comp, 'gene_i': t.index[i], 'gene_j': t.index[j],
                                  'peak_i': peaks[i], 'peak_j': peaks[j], 'peak separation': sep_obs[k],
                                  'dissimilarity': dis_obs[k],
                                  'separation percentile': (np.sum(sep_null[k] < sep_obs[k]) + .5 * np.sum(sep_null[k] == sep_obs[k])) / N_DRAWS,
                                  'dissimilarity percentile': (np.sum(dis_null[k] < dis_obs[k]) + .5 * np.sum(dis_null[k] == dis_obs[k])) / N_DRAWS,
                                  'replicated separation': separated, 'null replicated separation rate': rep_null[k].mean(),
                                  'peak segments i': f'{t.peak_segment_1.iloc[i]}/{t.peak_segment_2.iloc[i]}',
                                  'peak segments j': f'{t.peak_segment_1.iloc[j]}/{t.peak_segment_2.iloc[j]}',
                                  'segment means separate': seg_diff,
                                  'expression ratio (log2)': t.log2_cp10k.iloc[i] - t.log2_cp10k.iloc[j],
                                  'probe status': f'{t.probe_status.iloc[i]}/{t.probe_status.iloc[j]}'})
    pairs_ = pd.DataFrame(pair_rows)
    units = pairs_.groupby('unit').agg(species=('species', 'first'), component=('component', 'first'),
                                       pairs=('gene_i', 'size'), separation_score=('separation percentile', 'mean'),
                                       dissimilarity_score=('dissimilarity percentile', 'mean'),
                                       mean_separation=('peak separation', 'mean'),
                                       mean_dissimilarity=('dissimilarity', 'mean'),
                                       replicated_separated_pairs=('replicated separation', 'sum'))
    tests = []
    for k, (stat, score, column) in enumerate((('peak separation', 'separation_score', 'mean_separation'),
                                               ('dissimilarity 1 − r', 'dissimilarity_score', 'mean_dissimilarity'))):
        observed = units[column].mean()
        null = np.mean([null_store[u][k] for u in units.index], axis=0)
        w = wilcoxon(units[score] - .5) if len(units) >= 5 else None
        tests.append({'probe rule': clean_col, 'statistic': stat, 'units': len(units),
                      'median unit percentile': units[score].median(),
                      'Wilcoxon p (two-sided)': w.pvalue if w is not None else np.nan,
                      'observed mean': observed, 'matched null mean': null.mean(),
                      'null 2.5%': np.quantile(null, .025), 'null 97.5%': np.quantile(null, .975),
                      'global p (two-sided)': pz.global_two_sided(observed, null)})
    global_null = {k: np.mean([null_store[u][k] for u in units.index], axis=0) for k in range(3)}
    return pairs_, units, pd.DataFrame(tests), global_null


pair_table, unit_scores, tests_b, global_null = partition_analysis('probe_clean', SEED)
rep_null_global = global_null[2]
pairs_strict, units_strict, tests_strict, _ = partition_analysis('probe_clean_strict', SEED + 2)
tests_b = pd.concat([tests_b, tests_strict], ignore_index=True)
tests_b['test run (pre-declared ≥ 8 units)'] = STAGE2['b_test']
print(f'Replicated separations: {int(pair_table["replicated separation"].sum())} of {len(pair_table)} paralog pairs '
      f'({pair_table["replicated separation"].mean():.2f}); matched random pairs {rep_null_global.mean():.2f} '
      f'(95% of draws {np.quantile(rep_null_global, .025):.2f}–{np.quantile(rep_null_global, .975):.2f}).')
sep = pair_table[pair_table['replicated separation']]
print(f'Of the replicated separations, label-segment means separate {int(sep["segment means separate"].sum())}; '
      f'{int((~sep["segment means separate"]).sum())} share a peak segment in at least one specimen (continuous only): '
      + ', '.join(f'{a}/{b}' for a, b in zip(sep.loc[~sep["segment means separate"], "gene_i"], sep.loc[~sep["segment means separate"], "gene_j"])))
display(tests_b.round(3))
display(unit_scores.round(2))

**Probe cross-talk check for the paralog pairs (added after the test, before interpretation).**
Co-positioned paralogs could share signal if a probe of one gene also hybridises the other's
transcript beyond the audit's 5-mismatch window. For each pair, the smallest ungapped Hamming
distance between any probe of one gene and any 50-nt window of the other gene's transcripts (both
directions) is computed directly. Genomic position (same chromosome, within 1 Mb) marks tandem pairs.
Both are descriptive.

In [ ]:
pair_genes = {sp: sorted(set(pair_table.loc[pair_table.species.eq(sp), 'gene_i']) | set(pair_table.loc[pair_table.species.eq(sp), 'gene_j']))
              for sp in SPECIMENS}
probe_seq_of = {sp: probe_sets[sp].groupby('gene_id').probe_seq.apply(list) for sp in SPECIMENS}
intervals_of = {}
distance_rows = []
for sp in SPECIMENS:
    ids = gene_tables[sp].loc[pair_genes[sp], 'gene_id']
    seqs = pz.transcript_sequences(TRANSCRIPTS[sp], ids.tolist())
    intervals_of[sp] = pz.gene_intervals(pz.transcript_table(TRANSCRIPTS[sp]))
    for r in pair_table[pair_table.species.eq(sp)].itertuples():
        gi, gj = ids[r.gene_i], ids[r.gene_j]
        d_ij = min((pz.min_window_mismatches(q, seqs.get(gj, [])) for q in probe_seq_of[sp].get(gi, [])), default=np.nan)
        d_ji = min((pz.min_window_mismatches(q, seqs.get(gi, [])) for q in probe_seq_of[sp].get(gj, [])), default=np.nan)
        iv = intervals_of[sp].reindex([gi, gj])
        same_chrom = iv.chrom.iloc[0] == iv.chrom.iloc[1]
        gap = abs(iv.start.iloc[0] - iv.start.iloc[1]) if same_chrom else np.nan
        distance_rows.append({'unit': r.unit, 'gene_i': r.gene_i, 'gene_j': r.gene_j,
                              'min probe mismatches i→j': d_ij, 'min probe mismatches j→i': d_ji,
                              'same chromosome': same_chrom, 'genomic distance (kb)': gap / 1e3 if same_chrom else np.nan,
                              'tandem (< 1 Mb)': bool(same_chrom and gap < 1e6)})
pair_table = pair_table.merge(pd.DataFrame(distance_rows), on=['unit', 'gene_i', 'gene_j'], how='left')
pair_table['min probe mismatches'] = pair_table[['min probe mismatches i→j', 'min probe mismatches j→i']].min(axis=1)
pair_table['co-positioned (1 − r < 0.2)'] = pair_table.dissimilarity < .2
display(pair_table.groupby('co-positioned (1 − r < 0.2)')[['min probe mismatches', 'tandem (< 1 Mb)']]
        .agg(['median', 'mean', 'size']).round(2))
close = pair_table[pair_table['min probe mismatches'] <= 8]
print(f'Pairs with a probe within 8 mismatches of the paralog transcript: {len(close)} of {len(pair_table)}: '
      + ', '.join(f'{a}/{b} ({m:.0f})' for a, b, m in zip(close.gene_i, close.gene_j, close['min probe mismatches'])))
keep_units = pair_table[pair_table['min probe mismatches'] > 8].groupby('unit').agg(
    mean_separation=('peak separation', 'mean'), separation_score=('separation percentile', 'mean'))
print(f'Without those pairs: {len(keep_units)} units, median separation percentile '
      f'{keep_units.separation_score.median():.2f}, Wilcoxon p = {wilcoxon(keep_units.separation_score - .5).pvalue:.3f}.')
pair_table.to_csv(tables / 'stage2b_paralog_pairs.csv', index=False)
unit_scores.to_csv(tables / 'stage2b_unit_scores.csv')
tests_b.to_csv(tables / 'stage2b_tests.csv', index=False)
display(pair_table[['unit', 'gene_i', 'gene_j', 'peak_i', 'peak_j', 'dissimilarity', 'separation percentile',
                    'replicated separation', 'segment means separate', 'probe status', 'min probe mismatches',
                    'tandem (< 1 Mb)']].round(2))

**Reading.** The pre-declared test says paralogs are **less** separated and **more** alike in shape than
matched random zonated genes. That is the co-regulation prior, not positional subfunctionalisation. The
direction holds under the strict probe rule (p = 0.05 and 0.012). Co-positioned and separated pairs have
the same median distance between a probe and the paralog's transcript (13 mismatches), so probe
cross-talk does not explain the similarity. Every pair is a tandem duplicate (< 1 Mb apart), so tandem
status cannot separate the two groups. Dropping the 13 pairs with a probe within 8 mismatches leaves the
median below 0.5 but no longer significant (post hoc). Several pairs with probes 6–7 mismatches from the
paralog are still sharply separated (Acaa1a/Acaa1b, Cyp2d12/Cyp2d9), which argues against strong
cross-hybridisation at that distance.

Where paralogs do separate (Cyp2d, Cyp2j, Akr1c, Aldh3b, Ceacam, Acaa1, Ces2, Ugt2b34 against
Ugt2b37/38, Cyp4a12a against Cyp4a10/31), the separation is between segments: label-segment means alone
find 16 of the 17 replicated separations. The one within-segment separation, Cyp4a12a (peak 0.36)
against Cyp4a14 (0.60), both in S2, is the only case where the continuous coordinate adds to segment
means here. Male microdissection puts Cyp4a12a highest in S3, which does not support it.

### 6b · Does the single-copy gene span its co-orthologs' positions? (union fit) · Stage 2(c) sequence match

For 1:many and many:1 families whose single-copy gene is PT-expressed and probe-clean, with ≥ 2
PT-expressed, probe-clean co-orthologs: the single-copy pooled profile (linear scale, unit mean) is
fitted by a non-negative combination of the co-orthologs' profiles. The union R² is compared with
1,000 random k-sets from the expansion species matched on expression quintile. Stage 2(c) asks which
co-ortholog matches the single-copy gene best in position (mean cross-species Pearson of φ over the
four specimen pairs) and whether it is the sequence-closest one (Ensembl Compara 98 identity).

In [ ]:
compara = pd.read_csv(COMPARA, sep='\t', usecols=['gene_stable_id', 'identity', 'homology_type',
                                                  'homology_gene_stable_id', 'homology_identity'])
compara['mean_identity'] = (compara.identity + compara.homology_identity) / 2
identity = compara.groupby(['gene_stable_id', 'homology_gene_stable_id']).mean_identity.max()
pooled_lin = {sp: np.mean([fits[n]['curve'] for n in SPECIMENS[sp]], axis=0) for sp in SPECIMENS}
union_rows = []
rng_u = np.random.default_rng(SEED + 1)
for r in catalogue.itertuples():
    for single_sp, multi_sp in (('human', 'mouse'), ('mouse', 'human')):
        t_s, t_m = gene_tables[single_sp], gene_tables[multi_sp]
        singles = t_s[t_s.component.eq(r.component)]
        multis = t_m[t_m.component.eq(r.component) & t_m.pt_expressed & t_m.probe_clean]
        cls = singles.ortholog_class.iloc[0] if len(singles) else ''
        wanted = '1:many' if single_sp == 'human' else 'many:1'
        if cls != wanted or len(singles) != 1 or len(multis) < 2:
            continue
        g = singles.index[0]
        if not (singles.pt_expressed.iloc[0] and singles.probe_clean.iloc[0]):
            continue
        y = pz.unit_mean(pooled_lin[single_sp][row_of[single_sp][g]][None])[0]
        idx_m = row_of[multi_sp].loc[multis.index].to_numpy()
        X = pz.unit_mean(pooled_lin[multi_sp][idx_m])
        union, coef, best_single, best = pz.nnls_union(y, X)
        # matched null: random k-sets from the expansion species, member-matched on expression quintile
        elig = (t_m.pt_expressed & t_m.probe_clean & ~t_m.component.eq(r.component)).to_numpy()
        eq = np.full(len(t_m), -1)
        eq[elig] = pd.qcut(t_m.log2_cp10k[elig], 5, labels=False).to_numpy()
        target_q = pd.qcut(t_m.log2_cp10k[t_m.pt_expressed & t_m.probe_clean], 5, labels=False).reindex(multis.index).to_numpy()
        null_r2 = []
        for _ in range(N_DRAWS):
            pick = [rng_u.choice(np.flatnonzero(eq == q)) for q in target_q]
            null_r2.append(pz.nnls_union(y, pz.unit_mean(pooled_lin[multi_sp][pick]))[0])
        null_r2 = np.array(null_r2)
        # expression match per co-ortholog across the four cross-species specimen pairs
        phi_single = [pz.log_profile(fits[n]['curve'][row_of[single_sp][g]], PSEUDOCOUNT) for n in SPECIMENS[single_sp]]
        match = {}
        for m_gene, i in zip(multis.index, idx_m):
            phi_m = [pz.log_profile(fits[n]['curve'][i], PSEUDOCOUNT) for n in SPECIMENS[multi_sp]]
            match[m_gene] = float(np.mean([pz.rowwise_pearson(a[None], b[None])[0] for a in phi_single for b in phi_m]))
        ident = {}
        for m_gene in multis.index:
            hid = singles.gene_id.iloc[0] if single_sp == 'human' else t_m.loc[m_gene, 'gene_id']
            mid = t_m.loc[m_gene, 'gene_id'] if single_sp == 'human' else singles.gene_id.iloc[0]
            ident[m_gene] = identity.get((hid, mid), np.nan)
        ident = pd.Series(ident)
        match = pd.Series(match)
        paper_partner = gene_tables[single_sp].loc[g, 'paper_partner']
        union_rows.append({
            'component': r.component, 'single-copy species': single_sp, 'single-copy gene': g,
            'single Z': float(singles.Z.iloc[0]), 'single peak': float(singles.peak.iloc[0]),
            'co-orthologs (PT-expressed, clean)': ', '.join(multis.index), 'k': len(multis),
            'union R2': union, 'best single R2': best_single, 'best single (shape)': multis.index[best],
            'union R2 null median': float(np.median(null_r2)), 'union R2 percentile': float(np.mean(null_r2 < union)),
            'NNLS weights': ', '.join(f'{m}:{c:.2f}' for m, c in zip(multis.index, coef)),
            'breadth single': pz.breadth(y), 'breadth co-ortholog sum': pz.breadth(X.sum(axis=0)),
            'breadth co-orthologs max': float(max(pz.breadth(x) for x in X)),
            'best expression match': match.idxmax(), 'best expression r': float(match.max()),
            'expression r by co-ortholog': ', '.join(f'{m}:{v:.2f}' for m, v in match.sort_values(ascending=False).items()),
            'sequence-closest': ident.idxmax() if ident.notna().sum() >= 2 else '',
            'identity by co-ortholog': ', '.join(f'{m}:{v:.0f}' for m, v in ident.dropna().sort_values(ascending=False).items()),
            'n with identity': int(ident.notna().sum()),
            'paper partner': paper_partner if isinstance(paper_partner, str) else '',
            'paper partner is best expression match': (paper_partner == match.idxmax()) if isinstance(paper_partner, str) else None,
        })
union_table = pd.DataFrame(union_rows)
union_table.to_csv(tables / 'stage2bc_union_and_sequence_match.csv', index=False)
if len(union_table):
    testable = union_table[union_table['sequence-closest'].ne('')]
    hits = int((testable['best expression match'] == testable['sequence-closest']).sum())
    chance = [1 / k for k in testable['n with identity']]
    # Poisson-binomial distribution of the number of chance agreements
    dist = np.array([1.])
    for p_ in chance:
        dist = np.convolve(dist, [1 - p_, p_])
    p_two = float(min(1, 2 * min(dist[:hits + 1].sum(), dist[hits:].sum())))
    print(f'Union fits: {len(union_table)} families; median union R² {union_table["union R2"].median():.2f} '
          f'(matched random sets {union_table["union R2 null median"].median():.2f}); union above the null 95th '
          f'percentile in {int((union_table["union R2 percentile"] >= .95).sum())}.')
    print(f'Expression match = sequence-closest in {hits} of {len(testable)} families; chance expectation '
          f'{sum(chance):.1f}; Poisson-binomial two-sided p = {p_two:.2f}.')
    known = union_table[union_table['paper partner'].ne('')]
    print(f'The paper map kept a partner in {len(known)} of these families; it is the best expression match in '
          f'{int(known["paper partner is best expression match"].sum())}.')
    zonated_single = union_table[union_table['single Z'] > 0]
    print(f'Post hoc, single-copy gene zonated ({len(zonated_single)} families): median union R² '
          f'{zonated_single["union R2"].median():.2f} against matched sets {zonated_single["union R2 null median"].median():.2f}; '
          f'above the null 95th percentile in {int((zonated_single["union R2 percentile"] >= .95).sum())}: '
          + ', '.join(zonated_single.loc[zonated_single['union R2 percentile'] >= .95, 'single-copy gene']))
    display(union_table[['single-copy gene', 'co-orthologs (PT-expressed, clean)', 'union R2', 'best single R2',
                         'union R2 percentile', 'breadth single', 'breadth co-ortholog sum', 'best expression match',
                         'sequence-closest', 'paper partner']])

**Reading.** The union hypothesis is not supported. Most single-copy genes in these families are flat,
so the union fit has nothing to explain (negative R² around the mean). Among the 5 zonated single-copy
genes only one (Ctsl) is fitted better than matched random sets (post hoc). Which co-ortholog tracks the
single-copy gene's position is unrelated to sequence identity. For the paper map, the reciprocal-best
partner is the best positional match in 20 of 34 such families.

## 7 · Stage 2(d) · Zonated genes without an ortholog

The pre-declared rule runs the niche analysis only when at least 10 zonated, probe-clean genes
without any HCOP ortholog exist in a species. The inventory found fewer (below), so the niche
analysis is not run; the genes are listed, with genes whose only HCOP links are low-support
(`no_consensus`) beside them for completeness.

In [ ]:
print('Stage 2(d) niche analysis runs:', STAGE2['d'])
specific_rows = []
for sp in SPECIMENS:
    t = gene_tables[sp]
    for cls in ('no_ortholog', 'no_consensus'):
        for g, r in t[t.ortholog_class.eq(cls) & t.zonated & t.probe_clean].sort_values('Z', ascending=False).iterrows():
            specific_rows.append({'species': sp, 'class': cls, 'gene': g, 'Z': round(r.Z, 2), 'peak': round(r.peak, 2),
                                  'peak segment': r.peak_segment, 'log2 cp10k': round(r.log2_cp10k, 2),
                                  'probe status': r.probe_status, 'max HCOP support': r.max_support,
                                  'flags': ', '.join(f for f, on in (('fasting/PPARα', bool(r.flag_fasting_ppara)),
                                                                     ('androgen', bool(r.flag_androgen))) if on)})
specific_table = pd.DataFrame(specific_rows)
specific_table.to_csv(tables / 'stage2d_species_specific_zonated.csv', index=False)
display(specific_table)
not_zonated_kap = gene_tables['mouse'].loc[['Kap']] if 'Kap' in gene_tables['mouse'].index else None
if not_zonated_kap is not None:
    display(not_zonated_kap[['ortholog_class', 'probe_status', 'pt_expressed', 'zonated', 'Z', 'amplitude_1',
                             'amplitude_2', 'replication_r', 'peak', 'log2_cp10k']])

**Reading.** Kap, a mouse-specific gene, is among the most abundant PT transcripts in our male mice
(about 640 CP10k) and peaks late (0.75; S3 > S2 ≫ S1 in microdissection, in both sexes). The remaining
zonated species-specific genes are low-abundance.

## 8 · External read-based references for the highlighted genes

Mouse: microdissected PTS1/PTS2/PTS3 (Chen et al. 2021, GSE150338, male replicates; Chen et al. 2023,
GSE212213, male and female means). Human: Lake et al. 2023 KPMP snRNA, normal donors, cortex nuclei
labelled PT-S1/S2/S3, donors with ≥ 50 nuclei per segment (notebook 38 rule), per-donor log2 CPM
against all genes. Read-based quantification discards or splits multi-mapping reads, so near-identical
paralogs can be under-counted there; the probe audit is the corresponding check on our side.

In [ ]:
from pseudospace.replication_rules import csr_group_sums

LAKE = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
MICRO = external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz'
MICRO_SEX = external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz'
for path in (LAKE, MICRO, MICRO_SEX, external / 'xiong2023_sex_bias' / 'PT-S1_fWTvsmWT.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')


def read_obs_column(group, name):
    item = group[name]
    if isinstance(item, h5py.Group):
        categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
        return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
    values = item[:]
    return values.astype(str) if values.dtype.kind in 'SO' else values


def lake_pseudobulk_all_genes():
    with h5py.File(LAKE, 'r') as handle:
        obs = pd.DataFrame({c: read_obs_column(handle['obs'], c)
                            for c in ['subclass.l2', 'disease', 'region', 'tissue', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        keep = (obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3']) & obs.disease.eq('normal')
                & obs.region.eq('C') & obs.tissue.eq('cortex of kidney'))
        label = (obs.donor_id + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups_, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, counts = csr_group_sums(raw['data'], raw['indices'], raw['indptr'],
                                              np.flatnonzero(keep.to_numpy()), codes, len(groups_), len(var_ids),
                                              block_rows=5000)
    return {'groups': groups_.astype(str), 'genes': var_ids, 'sums': sums, 'totals': totals, 'counts': counts}


lake = cached_payload('lake_cortex_pt_all_genes', lake_pseudobulk_all_genes, root=cache_root,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'disease': 'normal', 'region': 'C, cortex of kidney'},
                      inputs={'lake_file': [LAKE.name, LAKE.stat().st_size]},
                      code=[lake_pseudobulk_all_genes, csr_group_sums])
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'segment'])
lake_meta['nuclei'] = lake['counts']
per_donor = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = per_donor.index[(per_donor[['S1', 'S2', 'S3']] >= 50).all(axis=1)]
lake_log = pd.DataFrame(np.log2(lake['sums'] / lake['totals'][:, None] * 1e6 + 1), columns=lake['genes'])
lake_seg = pd.DataFrame({s: lake_log[(lake_meta.segment.eq(s) & lake_meta.donor.isin(lake_donors)).to_numpy()].mean(axis=0)
                         for s in ('S1', 'S2', 'S3')})
lake_cpm_max = pd.DataFrame({s: (lake['sums'] / lake['totals'][:, None] * 1e6)[(lake_meta.segment.eq(s)
                             & lake_meta.donor.isin(lake_donors)).to_numpy()].mean(axis=0) for s in ('S1', 'S2', 'S3')},
                            index=lake['genes']).max(axis=1)
print(f'Lake cortex: {len(lake_donors)} donors with ≥ 50 nuclei in each PT segment.')

micro = pd.read_csv(MICRO, sep='\t')
micro = micro.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_male = pd.DataFrame({s: micro.filter(regex=rf'_PTS{k}_').mean(axis=1) for k, s in ((1, 'S1'), (2, 'S2'), (3, 'S3'))})
micro_sex = pd.read_csv(MICRO_SEX, sep='\t').drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_m2 = micro_sex[['PTS1', 'PTS2', 'PTS3']].set_axis(['S1', 'S2', 'S3'], axis=1)
micro_f = micro_sex[['fPTS1', 'fPTS2', 'fPTS3']].set_axis(['S1', 'S2', 'S3'], axis=1)
# Xiong et al. 2023 (PMID 37673062): female vs male wild-type mouse PT snRNA, per segment.
XIONG = external / 'xiong2023_sex_bias'
xiong = {s: pd.read_csv(XIONG / f'PT-{s}_fWTvsmWT.csv').drop_duplicates('symbol').set_index('symbol') for s in ('S1', 'S2', 'S3')}


def external_row(species, gene):
    r = gene_tables[species].loc[gene]
    out = {'species': species, 'gene': gene, 'ours peak segment': r.peak_segment, 'ours peak': r.peak, 'Z': r.Z,
           'probe status': r.probe_status}
    if species == 'mouse':
        fc = {s: xiong[s].loc[gene] if gene in xiong[s].index else None for s in xiong}
        out['Xiong female−male log2FC S1/S2/S3'] = '/'.join('na' if v is None or not bool(v.tested) else f'{v.log2FC:.1f}'
                                                         for v in fc.values())
        out['male-biased (padj < 0.05, log2FC < −1) in'] = ','.join(
            s for s, v in fc.items() if v is not None and bool(v.significant) and v.log2FC < -1)
        for name, frame in (('micro male 2021', micro_male), ('micro male 2023', micro_m2), ('micro female 2023', micro_f)):
            if gene in frame.index and frame.loc[gene].max() >= 5:
                v = frame.loc[gene]
                out[f'{name} TPM S1/S2/S3'] = '/'.join(f'{x:.0f}' for x in v)
                out[f'{name} peak'] = v.idxmax()
            else:
                out[f'{name} TPM S1/S2/S3'] = 'not detected' if gene in frame.index else 'absent'
                out[f'{name} peak'] = ''
    else:
        gid = r.gene_id
        if gid in lake_seg.index and lake_cpm_max.get(gid, 0) >= 10:
            v = lake_seg.loc[gid]
            out['Lake log2 CPM S1/S2/S3'] = '/'.join(f'{x:.1f}' for x in v)
            out['Lake peak'] = v.idxmax()
        else:
            out['Lake log2 CPM S1/S2/S3'] = 'not detected' if gid in lake_seg.index else 'absent'
            out['Lake peak'] = ''
    return out

# Highlighted genes: members of stage-2(a) called families, of families with a replicated paralog
# separation, of the requested example families, and the zonated genes without an ortholog.
highlight_components = (set(family_units.loc[family_units['call species'], 'component'])
                        | set(pair_table.loc[pair_table['replicated separation'], 'component'])
                        | {int(gene_tables['mouse'].loc[g, 'component']) for g in
                           ['Cyp4a14', 'Slco1a1', 'Ugt2b35', 'Ces1d', 'Akr1c20', 'Gsta3']})
external_rows = []
for sp in SPECIMENS:
    t = gene_tables[sp]
    chosen = t[(t.component.isin(highlight_components) & t.pt_expressed)
               | (t.ortholog_class.eq('no_ortholog') & t.zonated & t.probe_clean)]
    for g in chosen.sort_values(['component', 'peak']).index:
        external_rows.append(external_row(sp, g) | {'component': int(t.loc[g, 'component'])})
external_table = pd.DataFrame(external_rows)
mouse_ext = external_table[external_table.species.eq('mouse') & external_table['micro male 2021 peak'].ne('')]
agree = (mouse_ext['ours peak segment'] == mouse_ext['micro male 2021 peak'])
human_ext = external_table[external_table.species.eq('human') & external_table['Lake peak'].ne('')]
agree_h = human_ext['ours peak segment'] == human_ext['Lake peak']
print(f'Peak segment agrees with male microdissection for {int(agree.sum())} of {len(mouse_ext)} mouse genes '
      f'(zonated: {int(agree[mouse_ext.Z > 0].sum())} of {int((mouse_ext.Z > 0).sum())}); with Lake for '
      f'{int(agree_h.sum())} of {len(human_ext)} human genes (zonated: {int(agree_h[human_ext.Z > 0].sum())} of {int((human_ext.Z > 0).sum())}).')
external_table.to_csv(tables / 'external_segment_checks.csv', index=False)
display(external_table)

## 9 · Figure 2 · Family heatmaps: paralog curves ordered by peak, species side by side

Rows are PT-expressed members, two thin strips per gene (one per specimen), ordered by the pooled
peak. Colour is each specimen's log2 curve relative to the gene's own mean over both specimens
(purple above, green below, grey = flat), so a flat gene reads as uniform grey whatever its level.
Grey labels mark probe-ambiguous genes, which carry no claim. Lines above each block show the
10th–90th percentile of that species' S1, S2 and S3 structures on the shared axis. Families shown
(presentation rule, fixed before plotting): the six requested example families, then families with a
stage-2(a) species call or a replicated paralog separation, ordered by the largest member Z.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

REQUESTED = ['Cyp4a14', 'Slco1a1', 'Ugt2b35', 'Ces1d', 'Akr1c20', 'Gsta3']
DIVERGING = LinearSegmentedColormap.from_list('rel', ['#1B7837', '#A6DBA0', '#F2F2F2', '#C2A5CF', '#762A83'])
seg_spans = {sp: {s: np.quantile(np.concatenate([native[n]['position'][native[n]['segment'] == s] for n in SPECIMENS[sp]]), [.1, .9])
                  for s in SEGMENTS} for sp in SPECIMENS}
requested_components = [int(gene_tables['mouse'].loc[g, 'component']) for g in REQUESTED]
extra = (set(family_units.loc[family_units['call species'], 'component'])
         | set(pair_table.loc[pair_table['replicated separation'], 'component'])) - set(requested_components)
max_z = {c: max(gene_tables[sp].loc[gene_tables[sp].component.eq(c), 'Z'].max() for sp in SPECIMENS) for c in extra}
shown = requested_components + sorted(extra, key=lambda c: -max_z[c])
shown = [c for c in shown if c != gene_tables['mouse'].loc['Clcnka', 'component']]  # thin-limb spillover, not a PT family


def family_block(ax, species, comp):
    t = gene_tables[species]
    s = t[t.component.eq(comp) & t.pt_expressed].sort_values('peak')
    if s.empty:
        ax.axis('off')
        ax.text(.5, .5, 'no PT-expressed member', transform=ax.transAxes, ha='center', va='center', fontsize=6.5, color=MUTED)
        return
    rows, labels, colors = [], [], []
    for g, r in s.iterrows():
        i = row_of[species][g]
        phi_g = np.stack([pz.log_profile(fits[n]['curve'][i], PSEUDOCOUNT) for n in SPECIMENS[species]])
        rows += list(phi_g - phi_g.mean())
        tag = '' if r.probe_clean else f' ({r.probe_status})'
        labels.append(f'{g}{tag}  Z {r.Z:.1f}' if r.zonated else f'{g}{tag}  not zonated')
        colors.append(INK if r.probe_clean else MUTED)
    image = ax.imshow(np.array(rows), aspect='auto', cmap=DIVERGING, norm=TwoSlopeNorm(0, -3, 3),
                      interpolation='nearest', extent=[grid[0], grid[-1], len(rows), 0])
    for k in range(1, len(s)):
        ax.axhline(2 * k, color='white', lw=1.5)
    ax.set_yticks(np.arange(len(s)) * 2 + 1, labels, fontsize=6.3)
    for tick, c in zip(ax.get_yticklabels(), colors):
        tick.set_color(c)
    ax.tick_params(length=0)
    for k, seg in enumerate(SEGMENTS):
        a, b = seg_spans[species][seg]
        ax.plot([a, b], [-.35 - .5 * k] * 2, color=SPECIES_COLOR[species], lw=1.6, alpha=.4 + .2 * k, clip_on=False)
        ax.text(b, -.35 - .5 * k, f' {seg[-2:]}', fontsize=5.5, va='center', color=MUTED, clip_on=False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_visible(False)
    return image


heights = [max(int((gene_tables[sp].component.eq(c) & gene_tables[sp].pt_expressed).sum()) for sp in SPECIMENS) + 1.2
           for c in shown]
fig, axes = plt.subplots(len(shown), 2, figsize=(10, .3 * sum(heights) + 1.8),
                         gridspec_kw={'height_ratios': heights, 'hspace': .95, 'wspace': .7})
image = None
for k, comp in enumerate(shown):
    row = catalogue.set_index('component').loc[comp]
    for j, sp in enumerate(('mouse', 'human')):
        image = family_block(axes[k, j], sp, comp) or image
        axes[k, j].set_xlim(grid[0], grid[-1])
        axes[k, j].set_xticks([] if k < len(shown) - 1 else [0.1, 0.3, 0.5, 0.7])
    title = f'{row["mouse members in panel"] or "—"}  ↔  {row["human members in panel"] or "—"}'
    axes[k, 0].set_title(title if len(title) < 110 else title[:107] + '…', loc='left', fontsize=7.5, x=0, pad=12)
axes[-1, 0].set_xlabel('PT coordinate (SCF13), mouse')
axes[-1, 1].set_xlabel('PT coordinate (SCF13), human')
colorbar = fig.colorbar(image, ax=axes[:, 1], fraction=.02, pad=.02, shrink=.25, anchor=(0, 1))
colorbar.set_label('log2 relative to the gene mean', fontsize=7)
save(fig, 'fig2_family_heatmaps')
plt.show()
plt.close(fig)

## 10 · Figure 3 · Do paralogs partition the PT axis?

(a) Observed mean peak separation and shape dissimilarity of paralog pairs (one value per family ×
species, averaged) against 1,000 matched random-pair draws. (b) Per-family percentile of the peak
separation among matched random pairs. (c) Example families: pooled curves (each gene scaled to its
maximum) on the shared axis. (d) Each paralog pair's peak separation against whether label-segment
means alone separate the pair.

In [ ]:
fig = plt.figure(figsize=(13, 8.2))
gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.15], hspace=.55, wspace=.45)
primary = tests_b[tests_b['probe rule'].eq('probe_clean')].set_index('statistic')
for k, (stat, column) in enumerate((('peak separation', 'mean_separation'), ('dissimilarity 1 − r', 'mean_dissimilarity'))):
    ax = fig.add_subplot(gs[0, k])
    row = primary.loc[stat]
    ax.hist(global_null[k], bins=30, color=MUTED, alpha=.55, lw=0, label='1,000 matched random-pair draws')
    ax.axvline(row['observed mean'], color=INK, lw=2.2, label='paralog pairs')
    ax.set_yticks([])
    ax.set_xlabel(f'mean {stat} per family')
    ax.set_title(f'({"ab"[k]}) {stat}: p = {row["global p (two-sided)"]:.3f}', loc='left')
    ax.spines['left'].set_visible(False)
    if k == 0:
        ax.legend(frameon=False, fontsize=6.5, loc='upper right')

ax = fig.add_subplot(gs[0, 2:])
order_u = unit_scores.sort_values('separation_score')
labels_u = []
for u, r in order_u.iterrows():
    sp, comp = u.split(':')
    genes_u = sorted(set(pair_table.loc[pair_table.unit.eq(u), 'gene_i']) | set(pair_table.loc[pair_table.unit.eq(u), 'gene_j']))
    roots = sorted({pz.symbol_root(g).title() if sp == 'mouse' else pz.symbol_root(g) for g in genes_u})
    labels_u.append('/'.join(roots))
ax.scatter(range(len(order_u)), order_u.separation_score, s=18 + 10 * order_u.pairs,
           c=[SPECIES_COLOR[x] for x in order_u.species], alpha=.85, lw=0)
ax.axhline(.5, color=MUTED, lw=1, ls='--')
ax.set_xticks(range(len(order_u)), labels_u, rotation=60, ha='right', fontsize=6.3)
ax.set_ylabel('peak separation percentile\namong matched random pairs')
ax.set_ylim(0, 1)
ax.grid(axis='y', color=GRID, lw=.6)
ax.set_title(f'(c) Families: median {unit_scores.separation_score.median():.2f} '
             f'(Wilcoxon p = {primary.loc["peak separation", "Wilcoxon p (two-sided)"]:.3f}); dot size = pairs', loc='left')

example_units = [u for u in ('mouse:2144', 'mouse:10161', 'mouse:3515', 'mouse:8127') if u in unit_scores.index]
palette = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#56B4E9', '#D55E00']
for k, u in enumerate(example_units[:3]):
    ax = fig.add_subplot(gs[1, k])
    sp, comp = u.split(':')
    t = gene_tables[sp]
    members_u = t[t.component.eq(int(comp)) & t.zonated & t.probe_clean].sort_values('peak')
    for spn in SEGMENTS:
        a, b = seg_spans[sp][spn]
        level = .04 + .045 * SEGMENTS.index(spn)
        ax.plot([a, b], [level] * 2, color=SPECIES_COLOR[sp], lw=2, alpha=.35 + .2 * SEGMENTS.index(spn))
        ax.text(b, level, f' {spn[-2:]}', fontsize=6, color=MUTED, va='center')
    for c, (g, r) in zip(palette, members_u.iterrows()):
        curve = np.maximum(pooled_lin[sp][row_of[sp][g]], 0)
        ax.plot(grid, curve / curve.max(), color=c, lw=2, label=f'{g} ({r.probe_status})' if r.probe_status != 'clean' else g)
    ax.legend(frameon=False, fontsize=6.3, loc='upper center', ncol=2, bbox_to_anchor=(.5, 1.0))
    ax.set_ylim(0, 1.45)
    ax.set_xlabel('PT coordinate (SCF13)')
    ax.set_ylabel('curve ÷ its maximum' if k == 0 else '')
    ax.set_title(f'({"def"[k]}) {pz.symbol_root(members_u.index[0]).title()} family, {sp}', loc='left')

ax = fig.add_subplot(gs[1, 3])
yes = pair_table['segment means separate']
ax.scatter(pair_table['peak separation'][yes], np.random.default_rng(1).uniform(.6, 1.4, yes.sum()), color=INK, s=14,
           label='segment means separate')
ax.scatter(pair_table['peak separation'][~yes], np.random.default_rng(2).uniform(-.4, .4, (~yes).sum()),
           color=MUTED, s=14, label='same peak segment')
for _, r in pair_table[pair_table['replicated separation'] & ~yes].iterrows():
    ax.annotate(f'{r.gene_i}/{r.gene_j}', (r['peak separation'], 0), xytext=(5, 14), textcoords='offset points',
                fontsize=6.5, color=INK, arrowprops={'arrowstyle': '-', 'color': MUTED})
ax.axvline(PEAK_SEPARATION, color=MUTED, ls=':', lw=1)
ax.set_yticks([0, 1], ['same segment', 'different\nsegments'])
ax.set_xlabel('peak separation (shared axis)')
ax.set_title('(g) Continuous vs segment means', loc='left')
save(fig, 'fig3_paralog_partitioning')
plt.show()
plt.close(fig)

**Post hoc: are the mouse-zonated family genes a male program?** Xiong et al. 2023 compared female
and male wild-type mouse PT nuclei per segment. Share of strongly zonated, probe-clean mouse genes that
are male-biased (adjusted p < 0.05 and female − male log2FC < −1 in any segment), by class. All our
specimens are male, and human PT shows little sex difference in drug-handling proteins (PMID 38711199).

In [ ]:
t = gene_tables['mouse']
strong_m = t[t.strong & t.probe_clean].copy()
strong_m['male_biased'] = [any(g in xiong[s].index and bool(xiong[s].loc[g, 'significant']) and xiong[s].loc[g, 'log2FC'] < -1
                               for s in xiong) for g in strong_m.index]
strong_m['tested'] = [any(g in xiong[s].index and bool(xiong[s].loc[g, 'tested']) for s in xiong) for g in strong_m.index]
strong_m = strong_m[strong_m.tested]
strong_m['group'] = np.where(strong_m.ortholog_class.eq('1:1'), '1:1', 'non-1:1')
sex_table = strong_m.groupby('group').male_biased.agg(['sum', 'size', 'mean'])
odds_sex, p_sex = fisher_exact([[int(sex_table.loc['non-1:1', 'sum']), int(sex_table.loc['non-1:1', 'size'] - sex_table.loc['non-1:1', 'sum'])],
                                [int(sex_table.loc['1:1', 'sum']), int(sex_table.loc['1:1', 'size'] - sex_table.loc['1:1', 'sum'])]])
display(sex_table)
print(f'Male-biased share among strongly zonated mouse genes: non-1:1 {sex_table.loc["non-1:1", "mean"]:.2f} '
      f'vs 1:1 {sex_table.loc["1:1", "mean"]:.2f}; odds ratio {odds_sex:.2f}, Fisher p = {p_sex:.2g}.')
male_families = strong_m[strong_m.male_biased & strong_m.group.eq('non-1:1')]
print('Male-biased strongly zonated non-1:1 genes:', ', '.join(f'{g} ({r.peak_segment})' for g, r in male_families.iterrows()))
sex_table.to_csv(tables / 'posthoc_male_bias_of_strong_genes.csv')

## 11 · Run manifest

In [ ]:
manifest = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
    'inputs': {'coordinate': str(COORDINATE.relative_to(results_root)), 'hcop': HCOP.name,
               'human_panel': panel_name, 'transcriptomes': TRANSCRIPTOME_RELEASE},
    'rules': {'support': PRIMARY_SUPPORT, 'support_sensitivity': SUPPORT_RULES, 'detection': DETECTION, 'q': Q_MAX,
              'min_amplitude': MIN_AMPLITUDE, 'min_replication': MIN_REPLICATION, 'strong': STRONG,
              'pseudocount': PSEUDOCOUNT, 'grid': [float(lo), float(hi), GRID_POINTS], 'knots': knots.tolist(),
              'probe_cross_mm': CROSS_MM, 'probe_near_mm': NEAR_MM, 'stop_families': STOP_FAMILIES,
              'min_partition_families': MIN_PARTITION_FAMILIES, 'min_specific': MIN_SPECIFIC,
              'separation_gap': SEPARATION_GAP, 'peak_separation': PEAK_SEPARATION, 'draws': N_DRAWS, 'seed': SEED},
    'stage1': {'inventory': inventory.to_dict(orient='records'), 'share': share.round(4).to_dict(orient='records'),
               'families_zonated_both': F_both, 'families_partition': F_partition,
               'no_ortholog_zonated': {sp: len(v) for sp, v in specific.items()}, 'stage2': STAGE2,
               'paper_map_audit': paper_audit.to_dict(orient='records')},
    'stage2a': summary_a.astype(str).to_dict(orient='records'),
    'stage2b': tests_b.round(4).to_dict(orient='records'),
    'stage2c_union_rows': int(len(union_table)),
}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('Wrote', sorted(p.name for p in tables.iterdir()))

**Reading.** On the mouse side, the peak segment of zonated highlighted genes agrees with male
microdissection for 32 of 46. Agreement includes the separated paralog pairs: Cyp2d26/Cyp2d12 in S1
against Cyp2d9 in S2–S3, Ugt2b34 in S1 against Ugt2b37 in S2, Akr1c21 against Akr1c14/Akr1c18,
Aldh3b1/Aldh3b3, Ceacam1/Ceacam2, Acaa1a/Acaa1b, Cyp2j13 in S2 and Slco1a1/Slco1a6 in S2. On the human
side, the peak segment agrees with Lake for 13 of 14 zonated genes. Many of the mouse S2 family genes are
male-only in the female microdissection (Slco1a1, Cyp2j13, Cyp4a12a, Ugt2b37 and the S2 part of
Ugt2b38) and male-biased in Xiong et al. 2023. The mouse zonation that the 1:1 space misses is therefore
partly an androgen-dependent S2 drug-handling program. Its absence in human PT may reflect species, sex
programs, or both; a male donor does not settle it, because human PT shows little sex difference in these
proteins (PMID 38711199).

**Literature checked for the highlighted families** (sources seen this round):

| Family | What is known | Our observation | Status |
|---|---|---|---|
| Cyp4a | Cyp4a12a is the predominant renal 20-HETE synthase, male- and androgen-induced; Cyp4a12b is barely detectable in kidney; females express Cyp4a10/Cyp4a14 ([PMID 17112342](https://pubmed.ncbi.nlm.nih.gov/17112342/)) | ordered positions Cyp4a12a 0.36 < Cyp4a14 0.60 < Cyp4a10/Cyp4a31 0.71; human CYP4A11 flat but probe-ambiguous with CYP4A22 | Sex and enzymology known. The within-S2 order is new and conflicts with microdissection for Cyp4a12a. |
| Kap | Expressed only in PT; pars recta responds to thyroid and sex hormones, PCT is androgen-dependent ([PMID 16150866](https://pubmed.ncbi.nlm.nih.gov/16150866/)) | Late PT, Z 2.6, very abundant | Known (consistent) |
| Ugt2b | Androsterone and 3α-diol are glucuronidated in male mouse kidney by Ugt2b37; castration represses it ([PMID 35336777](https://pubmed.ncbi.nlm.nih.gov/35336777/)) | Ugt2b37/38 in S2, Ugt2b34 in S1; human UGT2B7 in S1 | Sex dependence known; the segment position (S2) is supported only by microdissection |
| Slc22a OATs | Mouse Oat5 (Slc22a19) is apical in PT and a functional, not genetic, ortholog of human OAT4 ([PMID 38627096](https://pubmed.ncbi.nlm.nih.gov/38627096/)); SLC22A24 is kidney-specialised for steroid-conjugate reabsorption ([PMID 31553721](https://pubmed.ncbi.nlm.nih.gov/31553721/)) | Slc22a19/Slc22a26 (mouse) and SLC22A24 (human) all peak late PT | Late position of SLC22A24 in human PT not found in the literature checked; candidate "same niche, lineage-specific paralogs" |
| Sex programs | AR directly controls dimorphic PT genes in mouse ([PMID 37673062](https://pubmed.ncbi.nlm.nih.gov/37673062/)); kidney DMET proteins: 46 sex-dependent in mouse, none significant in human ([PMID 38711199](https://pubmed.ncbi.nlm.nih.gov/38711199/)) | Male bias enriched among zonated non-1:1 mouse genes | Supports the caveat |
| Cyp2d, Cyp2j, Nat8f, Akr1c, Ces1, Slco1a | No segment-resolved literature found in this round | Microdissection agrees with our peak segments | Not checked beyond microdissection |

## 12 · What this means for the paper

- **Methods hygiene (actionable).** The paper calls its space "one-to-one"; it is a reciprocal-best
  map that keeps one pair inside many families. Report this, and the audit numbers above. Gene-level
  claims about family members must say which paralog was kept: for example CYP4A11 is paired with
  Cyp4a31, and SLCO1A2 with Slco1a5, which is not PT-expressed.
- **Biology (supporting, one paragraph).** The strongest mouse PT zonation outside the shared space
  is in rodent-expanded drug-handling families. Many of these genes are male-biased and S2-centred,
  and their human counterparts are flat or absent. This is consistent with the paper's central claim
  that human PT is flatter, and it adds that part of the mouse excess is invisible to a 1:1 analysis.
- **Continuous-specific value is small here.** Paralogs co-position rather than partition, and segment
  means already separate 16 of the 17 reproducible paralog separations. This is a negative result for
  the "continuous reveals subfunctionalisation" idea.

## 13 · Ranked findings (workstream C hand-off)

Each item gives numbers, specimen replication, probe status, external support, literature status and
the biggest caveat.

### 1. Male-biased drug-handling program outside the 1:1 space (most interesting; supporting point, not a headline)

The strongest mouse zonation outside the 1:1 space is a male-biased, mostly S1/S2 drug-handling program
in rodent-expanded families. The human members are flat or absent.

- **Numbers.**
  - 68 of 612 strongly zonated mouse genes are non-1:1 (odds ratio 2.1; expression-matched 2.3; 1.7–3.2
    across variants). 42 of the 68 are outside the shared space.
  - Their human orthogroup members: zonated 14, PT-expressed but flat 19, probe-ambiguous only 5,
    not PT-expressed 11, no HCOP ortholog 14, not on the panel 4.
  - Leads, with amplitude per mouse: Ugt2b38 8.1/8.4, Ugt2b37 6.0/6.6, Cyp2d26 6.4/6.7, Cyp2d12 5.9/5.9,
    Slc22a19 5.7/6.2, Akr1c18 5.6/6.0, Ces1e 5.3/5.5, Nat8f5 5.0/5.3, Cyp2j13 4.7/4.8, Slco1a6 4.5/4.9
    and Slco1a1 4.4/6.1. Replication r is 0.97–1.00.
- **Probes.** Clean, except Ugt2b37 and Ugt2b38, which are "near". The strict probe rule gives OR 2.1.
- **External support.**
  - Male microdissection agrees on the peak segment for 32 of 46 zonated highlighted genes.
  - Post hoc: 27% of these mouse genes are male-biased in Xiong 2023, against 16% of strongly zonated
    1:1 genes (p = 0.025).
  - In females, Ugt2b38 peaks in S1, like human UGT2B7, which peaks in S1 in our data and in Lake.
- **Literature.**
  - Known: the sex dependence of Cyp4a12a (PMID 17112342) and Ugt2b37 (PMID 35336777).
  - Mouse kidney drug-handling proteins are sex-dependent; human ones are not (PMID 38711199).
  - The segment positions of Cyp2d, Ugt2b38/37, Nat8f and Ces1e are not reported in the literature
    checked.
- **Caveat.** Sex programs may drive much of the family-level species difference. All specimens are male.

### 2. Paralogs co-position rather than partition the axis (negative for the continuous-specific idea)

Paralogs sit closer together on the axis than random genes, and segment means capture almost every
separation that does exist.

- **Numbers.**
  - Mean peak separation is 0.23, against 0.36 for matched random pairs (p = 0.02).
  - Shape dissimilarity (1 − r) is 0.61, against 0.93 (p = 0.014).
  - 17 of 58 pairs separate reproducibly, against 49% of matched random pairs.
  - Segment means also separate 16 of those 17. Only Cyp4a12a against Cyp4a14 (within S2) needs the
    continuous coordinate, and male microdissection contradicts it.
- **Probes.** The strict rule gives the same direction (p = 0.05 and 0.012). Dropping pairs with a probe
  within 8 mismatches of the paralog gives p = 0.145.
- **Literature.** The positions are not described.
- **Caveat.** All pairs are tandem duplicates.

### 3. The shared space is reciprocal-best, not one-to-one (actionable methods point)

- **Numbers.**
  - 1,051 of the 17,449 pairs lie in multi-member orthogroups.
  - In mouse, 59 of 393 expressed families keep a partner that is not their most-expressed PT member,
    and 29 keep a partner that is not PT-expressed at all. One example is SLCO1A2–Slco1a5, while Slco1a1
    and Slco1a6 carry the PT signal.
  - Expression match against sequence identity: 14 of 26 families, against 12.7 by chance.
- **Caveat.** Whether any paper claim depends on such a pick has not been checked.

### 4. A shared late-PT OAT niche held by different paralogs (candidate)

- **Numbers.** SLC22A24 (human, peak 0.79, amplitude 3.2/3.7) and Slc22a19/Slc22a26 (mouse, peak 0.78)
  sit in one many:many family.
- **Probes.** All three are clean.
- **External support.** Lake shows SLC22A24 rising to S3. In microdissection, Slc22a19 is in S2–S3.
- **Literature.**
  - Oat5 is a functional analogue of OAT4 (PMID 38627096).
  - SLC22A24 reabsorbs steroid conjugates in kidney (PMID 31553721).
  - No report places SLC22A24 in late PT.
- **Caveat.** One human donor.

### 5. Kap is mouse-specific, very abundant and late (known)

- **Numbers.** About 650 CP10k, peak 0.75, amplitude 2.6/3.9.
- **External support.** Microdissection gives S3 > S2 ≫ S1 in both sexes.
- **Literature.** Known (PMID 16150866).

### Verdict

This is a Supplementary Note plus two sentences, not a main section.

- **Methods.** Describe the shared space as reciprocal-best HCOP pairs and add the audit numbers.
- **Discussion.** Add one paragraph: the mouse zonation that the 1:1 space misses is concentrated in
  male-biased, rodent-expanded drug-handling families. This fits "human PT is flatter" but adds a sex
  confound that the male-only cohort cannot resolve.
- **Continuous coordinate.** It adds almost nothing for paralogs: they co-position, and segment means
  capture 16 of 17 separations.
- **Reusable method.** The probe audit can be reused for any paralog claim.